In [1]:
import pandas as pd
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

from src.utils.path import RAW_STATE_NY_PATH
df_raw = pd.read_csv(RAW_STATE_NY_PATH,dtype='string',keep_default_na=False)

assert ( df_raw.shape[1] == 99)
assert ( df_raw.columns.is_unique)
df_raw.shape

(383614, 99)

In [2]:
TARGET_MAPPING = {
    '1': 1,
    '2': 1,
    '3': 0,
}

eligiable_mask = df_raw['action_taken'].isin(TARGET_MAPPING)

dataset_b = df_raw.loc[eligiable_mask].copy()
dataset_b['y_approved'] = dataset_b['action_taken'].map(TARGET_MAPPING)

cohort_summary = pd.Series({
    'raw_rows':len(df_raw),
    'dataset_b_rows':len(dataset_b),
    'excluded_row':int((~eligiable_mask).sum())
})

cohort_summary

raw_rows          383614
dataset_b_rows    293301
excluded_row       90313
dtype: int64

In [3]:
assert dataset_b['y_approved'].notna().all()
assert set(dataset_b['action_taken'].unique()) == {'1','2','3'}
assert set(dataset_b['y_approved'].unique()) == {0,1}
assert dataset_b.loc[
    dataset_b['action_taken'].isin(['1','2']),
    'y_approved',
].eq(1).all()
assert dataset_b.loc[
    dataset_b["action_taken"].eq("3"),
    "y_approved",
].eq(0).all()

target_check = pd.crosstab(
    dataset_b["action_taken"],
    dataset_b["y_approved"],
    margins=True,
)

target_check

y_approved,0,1,All
action_taken,,,
1,0,208818,208818
2,0,9942,9942
3,74541,0,74541
All,74541,218760,293301


## 5. Feature cần kiểm tra thêm

### 5.1. Kênh nộp hồ sơ và quan hệ tổ chức

Nhóm này kiểm tra bốn candidate features:

- `preapproval`
- `submission_of_application`
- `initially_payable_to_institution`
- `conforming_loan_limit`

Mục tiêu là xác định liệu các feature có cung cấp thông tin hợp lệ,
có trước quyết định và khác với nhóm feature ưu tiên, hay chủ yếu phản
ánh quy trình tiếp nhận, cách tổ chức báo cáo hoặc danh tính tổ chức.

Các kiểm tra trong phần này gồm:

1. raw-token inventory và valid-code coverage;
2. informative, not-applicable, `Exempt`, undetermined và blank;
3. category support và mức độ mất cân bằng;
4. association trong block và với các feature ưu tiên liên quan;
5. concentration theo `lei` như organization-proxy diagnostic.

Phần này chưa thực hiện preprocessing, imputation, encoding, model
training, protected-group sensitivity hoặc quyết định feature cuối cùng.
Không sử dụng `lei` làm predictor.

In [4]:
import numpy as np
from itertools import combinations

GROUP_5_1_COLUMNS = [
    "preapproval",
    "submission_of_application",
    "initially_payable_to_institution",
    "conforming_loan_limit",
]

CONTEXT_5_1_COLUMNS = [
    "loan_purpose",
    "loan_type",
    "lien_status",
    "loan_amount",
]

REFERENCE_5_1_COLUMNS = [
    "lei",
]

required_columns_5_1 = (
    GROUP_5_1_COLUMNS
    + CONTEXT_5_1_COLUMNS
    + REFERENCE_5_1_COLUMNS
)

missing_columns_5_1 = [
    column
    for column in required_columns_5_1
    if column not in dataset_b.columns
]

assert not missing_columns_5_1, (
    f"Missing columns: {missing_columns_5_1}"
)

group_5_1_data = dataset_b.loc[
    :,
    required_columns_5_1,
].copy()

raw_token_inventory_5_1 = pd.concat(
    [
        (
            group_5_1_data[column]
            .value_counts(dropna=False)
            .rename_axis("raw_code")
            .reset_index(name="n")
            .assign(feature=column)
        )
        for column in GROUP_5_1_COLUMNS
    ],
    ignore_index=True,
)[
    ["feature", "raw_code", "n"]
]

raw_token_inventory_5_1

,feature,raw_code,n
0,preapproval,2,283785
1,preapproval,1,9516
2,submission_of_application,1,245493
3,submission_of_application,2,39534
4,submission_of_application,1111,8274
5,initially_payable_to_institution,1,269984
6,initially_payable_to_institution,2,9510
7,initially_payable_to_institution,1111,8274
8,initially_payable_to_institution,3,5533
9,conforming_loan_limit,C,280700


In [5]:
EXPECTED_CODE_LABELS_5_1 = {
    "preapproval": {
        "1": "Preapproval requested",
        "2": "Preapproval not requested",
        "3": "Not applicable",
    },
    "submission_of_application": {
        "1": "Submitted directly to reporting institution",
        "2": "Not submitted directly to reporting institution",
        "3": "Not applicable",
        "1111": "Exempt",
    },
    "initially_payable_to_institution": {
        "1": "Initially payable to reporting institution",
        "2": "Not initially payable to reporting institution",
        "3": "Not applicable",
        "1111": "Exempt",
    },
    "conforming_loan_limit": {
        "C": "Conforming",
        "NC": "Non-conforming",
        "U": "Undetermined",
    },
}

SPECIAL_CODES_5_1 = {
    "preapproval": {"3"},
    "submission_of_application": {"3", "1111"},
    "initially_payable_to_institution": {"3", "1111"},
    "conforming_loan_limit": {"U"},
}

In [6]:
def audit_feature_codes_5_1(data, feature):
    values = data[feature]

    code_labels = EXPECTED_CODE_LABELS_5_1[feature]
    special_codes = SPECIAL_CODES_5_1[feature]

    audit_rows = []

    for raw_code, count in values.value_counts(
        dropna=False
    ).items():
        blank_or_missing = (
            pd.isna(raw_code)
            or str(raw_code).strip() == ""
        )

        if blank_or_missing:
            description = "Blank or missing"
            status = "blank_or_missing"
        elif raw_code not in code_labels:
            description = "Unrecognized code"
            status = "invalid_code"
        elif raw_code in special_codes:
            description = code_labels[raw_code]
            status = "valid_special"
        else:
            description = code_labels[raw_code]
            status = "valid_informative"

        audit_rows.append({
            "feature": feature,
            "raw_code": raw_code,
            "description": description,
            "status": status,
            "n": int(count),
            "share_pct": round(
                count / len(values) * 100,
                4,
            ),
        })

    return pd.DataFrame(audit_rows)


audit_5_1 = pd.concat(
    [
        audit_feature_codes_5_1(
            data=group_5_1_data,
            feature=feature,
        )
        for feature in GROUP_5_1_COLUMNS
    ],
    ignore_index=True,
)

quality_rows_5_1 = []

for feature in GROUP_5_1_COLUMNS:
    feature_audit = audit_5_1.loc[
        audit_5_1["feature"] == feature
    ]

    informative_n = int(
        feature_audit.loc[
            feature_audit["status"] == "valid_informative",
            "n",
        ].sum()
    )

    special_n = int(
        feature_audit.loc[
            feature_audit["status"] == "valid_special",
            "n",
        ].sum()
    )

    blank_or_missing_n = int(
        feature_audit.loc[
            feature_audit["status"] == "blank_or_missing",
            "n",
        ].sum()
    )

    invalid_code_n = int(
        feature_audit.loc[
            feature_audit["status"] == "invalid_code",
            "n",
        ].sum()
    )

    valid_n = informative_n + special_n

    quality_rows_5_1.append({
        "feature": feature,
        "row_count": len(group_5_1_data),
        "valid_n": valid_n,
        "valid_pct": round(
            valid_n / len(group_5_1_data) * 100,
            4,
        ),
        "informative_n": informative_n,
        "informative_pct": round(
            informative_n / len(group_5_1_data) * 100,
            4,
        ),
        "special_n": special_n,
        "special_pct": round(
            special_n / len(group_5_1_data) * 100,
            4,
        ),
        "blank_or_missing_n": blank_or_missing_n,
        "invalid_code_n": invalid_code_n,
        "raw_unique_count": (
            group_5_1_data[feature]
            .nunique(dropna=False)
        ),
    })

quality_5_1 = pd.DataFrame(quality_rows_5_1)

invalid_codes_5_1 = audit_5_1.loc[
    audit_5_1["status"] == "invalid_code"
].copy()

display(quality_5_1)
display(audit_5_1)
display(invalid_codes_5_1)

,feature,row_count,valid_n,valid_pct,informative_n,informative_pct,special_n,special_pct,blank_or_missing_n,invalid_code_n,raw_unique_count
0,preapproval,293301,293301,100.0000,293301,100.0000,0,0.0000,0,0,2
1,submission_of_application,293301,293301,100.0000,285027,97.1790,8274,2.8210,0,0,3
2,initially_payable_to_institution,293301,293301,100.0000,279494,95.2925,13807,4.7075,0,0,4
3,conforming_loan_limit,293301,290194,98.9407,290136,98.9209,58,0.0198,0,3107,4


,feature,raw_code,description,status,n,share_pct
0,preapproval,2,Preapproval not requested,valid_informative,283785,96.7556
1,preapproval,1,Preapproval requested,valid_informative,9516,3.2444
2,submission_of_application,1,Submitted directly to reporting institution,valid_informative,245493,83.7000
3,submission_of_application,2,Not submitted directly to reporting institution,valid_informative,39534,13.4790
4,submission_of_application,1111,Exempt,valid_special,8274,2.8210
5,initially_payable_to_institution,1,Initially payable to reporting institution,valid_informative,269984,92.0501
6,initially_payable_to_institution,2,Not initially payable to reporting institution,valid_informative,9510,3.2424
7,initially_payable_to_institution,1111,Exempt,valid_special,8274,2.8210
8,initially_payable_to_institution,3,Not applicable,valid_special,5533,1.8865
9,conforming_loan_limit,C,Conforming,valid_informative,280700,95.7037


,feature,raw_code,description,status,n,share_pct
11,conforming_loan_limit,NA,Unrecognized code,invalid_code,3107,1.0593


In [7]:
assert len(group_5_1_data) == len(dataset_b)

assert quality_5_1["row_count"].eq(
    len(dataset_b)
).all()

quality_reconciled_5_1 = (
    quality_5_1["valid_n"]
    + quality_5_1["blank_or_missing_n"]
    + quality_5_1["invalid_code_n"]
)

assert quality_reconciled_5_1.eq(
    len(dataset_b)
).all()

audit_totals_5_1 = (
    audit_5_1
    .groupby("feature")["n"]
    .sum()
)

assert audit_totals_5_1.eq(
    len(dataset_b)
).all()

validation_5_1 = pd.Series({
    "row_count_reconciled": True,
    "quality_counts_reconciled": True,
    "audit_counts_reconciled": True,
    "schema_map_matches_observed_tokens": (
        invalid_codes_5_1.empty
    ),
})

validation_5_1

row_count_reconciled                   True
quality_counts_reconciled              True
audit_counts_reconciled                True
schema_map_matches_observed_tokens    False
dtype: bool

In [8]:
def cramers_v_5_1(data,left,right):
    contingency = pd.crosstab(
        data[left],
        data[right],
        dropna=False,
    )

    observed = contingency.to_numpy(
        dtype=float
    )

    n = observed.sum()
    rows,columns = observed.shape

    if(
        n == 0
        or rows < 2
        or columns < 2
    ):
        return np.nan

    expected = np.outer(
        observed.sum(axis= 1),
        observed.sum(axis= 0),
    ) /n 

    chi_square_terms = np.divide(
        (observed - expected) ** 2,
        expected,
        out=np.zeros_like(expected),
        where=expected > 0,
    )

    chi_square = chi_square_terms.sum()

    denominator = min(rows -1, columns - 1,)
    if denominator == 0:
        return np.nan

    return float(
        np.sqrt(
            (chi_square/ n)
            / denominator
        )
    )

association_specs_5_1 = [
    (left,right,'within_5_1_block')
    for left,right in combinations(
        GROUP_5_1_COLUMNS,
        2,
    )
]

association_specs_5_1.extend([
    (
        "preapproval",
        "loan_purpose",
        "priority_feature_context",
    ),
    (
        "conforming_loan_limit",
        "loan_type",
        "priority_feature_context",
    ),
    (
        "conforming_loan_limit",
        "lien_status",
        "priority_feature_context",
    ),
])

association_5_1 = pd.DataFrame(
    [
        {
            "left_feature": left,
            "right_feature": right,
            "scope": scope,
            "cramers_v": round(
                cramers_v_5_1(
                    group_5_1_data,
                    left,
                    right,
                ),
                4,
            ),
        }
        for left, right, scope
        in association_specs_5_1
    ]
)

association_5_1

,left_feature,right_feature,scope,cramers_v
0,preapproval,submission_of_application,within_5_1_block,0.0633
1,preapproval,initially_payable_to_institution,within_5_1_block,0.0288
2,preapproval,conforming_loan_limit,within_5_1_block,0.0560
3,submission_of_application,initially_payable_to_institution,within_5_1_block,0.7178
4,submission_of_application,conforming_loan_limit,within_5_1_block,0.0654
5,initially_payable_to_institution,conforming_loan_limit,within_5_1_block,0.0497
6,preapproval,loan_purpose,priority_feature_context,0.1927
7,conforming_loan_limit,loan_type,priority_feature_context,0.0363
8,conforming_loan_limit,lien_status,priority_feature_context,0.0890


In [9]:
def row_percentage_crosstab_5_1(
    data,
    row_feature,
    column_feature,
):
    counts = pd.crosstab(
        data[row_feature],
        data[column_feature],
        dropna=False,
    )

    row_pct = (
        counts
        .div(counts.sum(axis=1), axis=0)
        .mul(100)
        .round(2)
    )

    return counts, row_pct


key_relationships_5_1 = [
    (
        "submission_of_application",
        "initially_payable_to_institution",
    ),
    (
        "preapproval",
        "loan_purpose",
    ),
    (
        "conforming_loan_limit",
        "loan_type",
    ),
    (
        "conforming_loan_limit",
        "lien_status",
    ),
]

relationship_tables_5_1 = {}

for left, right in key_relationships_5_1:
    counts, row_pct = row_percentage_crosstab_5_1(
        group_5_1_data,
        left,
        right,
    )

    relationship_tables_5_1[
        (left, right)
    ] = {
        "counts": counts,
        "row_pct": row_pct,
    }

    print(f"{left} × {right}")
    display(counts)
    display(row_pct)

submission_of_application × initially_payable_to_institution


initially_payable_to_institution,1,1111,2,3
submission_of_application,,,,
1,236015,0,5184,4294
1111,0,8274,0,0
2,33969,0,4326,1239


initially_payable_to_institution,1,1111,2,3
submission_of_application,,,,
1,96.14,0.0,2.11,1.75
1111,0.00,100.0,0.00,0.00
2,85.92,0.0,10.94,3.13


preapproval × loan_purpose


loan_purpose,1,2,31,32,4,5
preapproval,,,,,,
1,9516,0,0,0,0,0
2,129628,49762,29412,34595,40360,28


loan_purpose,1,2,31,32,4,5
preapproval,,,,,,
1,100.00,0.00,0.00,0.00,0.00,0.00
2,45.68,17.54,10.36,12.19,14.22,0.01


... (2 outputs left out)

loan_type,1,2,3,4
conforming_loan_limit,,,,
C,90.18,7.51,2.20,0.12
NA,99.71,0.29,0.00,0.00
NC,99.05,0.47,0.49,0.00
U,100.00,0.00,0.00,0.00


conforming_loan_limit × lien_status


lien_status,1,2
conforming_loan_limit,,
C,193289,87411
NA,3008,99
NC,8082,1354
U,52,6


lien_status,1,2
conforming_loan_limit,,
C,68.86,31.14
NA,96.81,3.19
NC,85.65,14.35
U,89.66,10.34


In [10]:
conforming_amount_data_5_1 = (
    group_5_1_data[
        [
            "conforming_loan_limit",
            "loan_amount",
        ]
    ]
    .copy()
)

conforming_amount_data_5_1[
    "loan_amount_numeric"
] = pd.to_numeric(
    conforming_amount_data_5_1[
        "loan_amount"
    ],
    errors="coerce",
)

conforming_amount_profile_5_1 = (
    conforming_amount_data_5_1
    .groupby(
        "conforming_loan_limit",
        dropna=False,
    )
    .agg(
        row_count=(
            "loan_amount",
            "size",
        ),
        numeric_n=(
            "loan_amount_numeric",
            "count",
        ),
        q1=(
            "loan_amount_numeric",
            lambda values: values.quantile(0.25),
        ),
        median=(
            "loan_amount_numeric",
            "median",
        ),
        q3=(
            "loan_amount_numeric",
            lambda values: values.quantile(0.75),
        ),
    )
    .reset_index()
)

conforming_amount_profile_5_1[
    "numeric_pct"
] = (
    conforming_amount_profile_5_1[
        "numeric_n"
    ]
    .div(
        conforming_amount_profile_5_1[
            "row_count"
        ]
    )
    .mul(100)
    .round(2)
)

conforming_amount_profile_5_1

,conforming_loan_limit,row_count,numeric_n,q1,median,q3,numeric_pct
0,C,280700,280700,95000.0,195000.0,395000.0,100.0
1,NA,3107,3107,805000.0,1825000.0,4405000.0,100.0
2,NC,9436,9436,1205000.0,1505000.0,2065000.0,100.0
3,U,58,58,825000.0,955000.0,1085000.0,100.0


In [11]:
def lei_proxy_diagnostic_5_1(
    data,
    features,
):
    lei_values = data["lei"]

    valid_lei_mask = (
        lei_values.notna()
        & lei_values.str.strip().ne("")
    )

    proxy_data = data.loc[
        valid_lei_mask,
        ["lei"] + features,
    ].copy()

    summary_rows = []
    detail_frames = []

    for feature in features:
        counts = (
            proxy_data
            .groupby(
                ["lei", feature],
                dropna=False,
            )
            .size()
            .rename("n")
            .reset_index()
        )

        counts["lei_total_n"] = (
            counts
            .groupby("lei")["n"]
            .transform("sum")
        )

        counts["within_lei_share"] = (
            counts["n"]
            / counts["lei_total_n"]
        )

        per_lei = (
            counts
            .groupby("lei")
            .agg(
                row_count=("n", "sum"),
                category_count=(
                    feature,
                    "nunique",
                ),
                dominant_share=(
                    "within_lei_share",
                    "max",
                ),
            )
            .reset_index()
        )

        per_lei.insert(
            0,
            "feature",
            feature,
        )

        detail_frames.append(per_lei)

        weighted_dominant_share = np.average(
            per_lei["dominant_share"],
            weights=per_lei["row_count"],
        )

        single_category_lei_n = int(
            per_lei["category_count"]
            .eq(1)
            .sum()
        )

        summary_rows.append({
            "feature": feature,
            "scope": "full_cohort_descriptive_only",
            "lei_n": len(per_lei),
            "blank_lei_row_n": int(
                (~valid_lei_mask).sum()
            ),
            "single_category_lei_n": (
                single_category_lei_n
            ),
            "single_category_lei_pct": round(
                single_category_lei_n
                / len(per_lei)
                * 100,
                2,
            ),
            "median_dominant_share_pct": round(
                per_lei["dominant_share"]
                .median()
                * 100,
                2,
            ),
            "row_weighted_dominant_share_pct": round(
                weighted_dominant_share
                * 100,
                2,
            ),
        })

    return (
        pd.DataFrame(summary_rows),
        pd.concat(
            detail_frames,
            ignore_index=True,
        ),
    )


(
    lei_proxy_summary_5_1,
    lei_proxy_detail_5_1,
) = lei_proxy_diagnostic_5_1(
    data=group_5_1_data,
    features=GROUP_5_1_COLUMNS,
)

lei_proxy_summary_5_1

,feature,scope,lei_n,blank_lei_row_n,single_category_lei_n,single_category_lei_pct,median_dominant_share_pct,row_weighted_dominant_share_pct
0,preapproval,full_cohort_descriptive_only,715,0,588,82.24,100.0,97.34
1,submission_of_application,full_cohort_descriptive_only,715,0,603,84.34,100.0,96.30
2,initially_payable_to_institution,full_cohort_descriptive_only,715,0,603,84.34,100.0,96.57
3,conforming_loan_limit,full_cohort_descriptive_only,715,0,377,52.73,100.0,95.93


In [12]:
quality_lookup_5_1 = (
    quality_5_1
    .set_index("feature")
)

decision_5_1 = pd.DataFrame({
    "feature_or_block": GROUP_5_1_COLUMNS,
})

decision_5_1["business_reason"] = (
    decision_5_1["feature_or_block"].map({
        "preapproval": (
            "Represents whether preapproval was requested."
        ),
        "submission_of_application": (
            "Represents the application submission channel."
        ),
        "initially_payable_to_institution": (
            "Represents the initial institutional relationship."
        ),
        "conforming_loan_limit": (
            "Represents reported conforming-loan-limit status."
        ),
    })
)

decision_5_1["timing_status"] = (
    "requires_lineage_confirmation"
)

decision_5_1["quality_evidence"] = (
    decision_5_1["feature_or_block"].map({
        feature: (
            f"{quality_lookup_5_1.loc[feature, 'valid_pct']:.2f}% "
            f"valid; "
            f"{quality_lookup_5_1.loc[feature, 'informative_pct']:.2f}% "
            f"informative; "
            f"{int(quality_lookup_5_1.loc[feature, 'invalid_code_n'])} "
            "invalid codes."
        )
        for feature in GROUP_5_1_COLUMNS
    })
)

decision_5_1["redundancy_evidence"] = (
    "Review association_5_1 and relationship_tables_5_1."
)

decision_5_1["organization_proxy_evidence"] = (
    "Review lei_proxy_summary_5_1; descriptive only."
)

decision_5_1["protected_group_sensitivity"] = (
    "not_run"
)

decision_5_1["screening_status"] = (
    "pending_train_validation_evidence"
)

decision_5_1["decision"] = pd.NA

decision_5_1["decision_reason"] = (
    "Pending lineage confirmation, train-only analysis, "
    "and grouped addition/ablation evidence."
)

decision_5_1["gsv_player"] = pd.NA

decision_5_1

,feature_or_block,business_reason,timing_status,quality_evidence,redundancy_evidence,organization_proxy_evidence,protected_group_sensitivity,screening_status,decision,decision_reason,gsv_player
0,preapproval,Represents whether preapproval was requested.,requires_lineage_confirmation,100.00% valid; 100.00% informative; 0 invalid ...,Review association_5_1 and relationship_tables...,Review lei_proxy_summary_5_1; descriptive only.,not_run,pending_train_validation_evidence,<NA>,"Pending lineage confirmation, train-only analy...",<NA>
1,submission_of_application,Represents the application submission channel.,requires_lineage_confirmation,100.00% valid; 97.18% informative; 0 invalid c...,Review association_5_1 and relationship_tables...,Review lei_proxy_summary_5_1; descriptive only.,not_run,pending_train_validation_evidence,<NA>,"Pending lineage confirmation, train-only analy...",<NA>
2,initially_payable_to_institution,Represents the initial institutional relations...,requires_lineage_confirmation,100.00% valid; 95.29% informative; 0 invalid c...,Review association_5_1 and relationship_tables...,Review lei_proxy_summary_5_1; descriptive only.,not_run,pending_train_validation_evidence,<NA>,"Pending lineage confirmation, train-only analy...",<NA>
3,conforming_loan_limit,Represents reported conforming-loan-limit status.,requires_lineage_confirmation,98.94% valid; 98.92% informative; 3107 invalid...,Review association_5_1 and relationship_tables...,Review lei_proxy_summary_5_1; descriptive only.,not_run,pending_train_validation_evidence,<NA>,"Pending lineage confirmation, train-only analy...",<NA>


### 5.1.1. Bổ sung bằng chứng để đóng screening

Các cell dưới đây **chỉ bổ sung** audit cho phần 5.1; không thay thế hoặc xóa các cell trước. Mục tiêu là sửa phân loại schema của `conforming_loan_limit`, tách association trên full population và informative-only population, làm rõ phạm vi áp dụng của `preapproval`, bổ sung LEI baseline/size sensitivity và tạo screening decision record.

Nguồn đối chiếu:

- `docs/handbook/contracts/feature_governance.md` và `docs/handbook/contracts/data_contract.md`: yêu cầu tách riêng `NA`, `Exempt`/`1111`, not-applicable; kiểm tra timing, conditional applicability, redundancy và organization proxy.
- [2024 HMDA Filing Instructions Guide](https://files.ffiec.cfpb.gov/documentation/2024-hmda-fig.pdf): định nghĩa `preapproval`, `submission_of_application` và `initially_payable_to_institution`.
- [CFPB Public LAR Code Sheet](https://s3.amazonaws.com/cfpb-hmda-public/prod/help/2018-public-LAR-code-sheet.pdf): `conforming_loan_limit` có bốn trạng thái hợp lệ `C`, `NC`, `U`, `NA`; `NA` là *Not Applicable*.
- [CFPB HMDA disclosure policy guidance](https://files.consumerfinance.gov/f/documents/HMDA_Data_Disclosure_Policy_Guidance.Executive_Summary.FINAL.12212018.pdf): conforming-loan-limit là public derived indicator dựa trên reported loan amount và GSE limit.

Ba trường về preapproval/application channel mô tả thông tin của yêu cầu hoặc hồ sơ. Riêng `conforming_loan_limit` là trường public-derived có thể dùng giới hạn tại thời điểm application **hoặc** origination; vì vậy chưa gắn nhãn an toàn trước quyết định cho mô hình approval cho đến khi lineage của từng action type được xác nhận.

In [13]:
EXPECTED_CODE_LABELS_5_1_REVIEWED = {
    **EXPECTED_CODE_LABELS_5_1,
    "conforming_loan_limit": {
        "C": "Conforming",
        "NC": "Non-conforming",
        "U": "Undetermined",
        "NA": "Not applicable",
    },
}

SPECIAL_CODES_5_1_REVIEWED = {
    **SPECIAL_CODES_5_1,
    "conforming_loan_limit": {"U", "NA"},
}

def audit_feature_codes_5_1_reviewed(data, feature):
    values = data[feature]
    code_labels = EXPECTED_CODE_LABELS_5_1_REVIEWED[feature]
    special_codes = SPECIAL_CODES_5_1_REVIEWED[feature]
    audit_rows = []

    for raw_code, count in values.value_counts(dropna=False).items():
        blank_or_missing = (
            pd.isna(raw_code)
            or str(raw_code).strip() == ""
        )

        if blank_or_missing:
            description = "Blank or missing"
            status = "blank_or_missing"
        elif raw_code not in code_labels:
            description = "Unrecognized code"
            status = "invalid_code"
        elif raw_code in special_codes:
            description = code_labels[raw_code]
            status = "valid_special"
        else:
            description = code_labels[raw_code]
            status = "valid_informative"

        audit_rows.append({
            "feature": feature,
            "raw_code": raw_code,
            "description": description,
            "status": status,
            "n": int(count),
            "share_pct": round(count / len(values) * 100, 4),
        })

    return pd.DataFrame(audit_rows)

audit_5_1_reviewed = pd.concat(
    [
        audit_feature_codes_5_1_reviewed(
            data=group_5_1_data,
            feature=feature,
        )
        for feature in GROUP_5_1_COLUMNS
    ],
    ignore_index=True,
)

quality_rows_5_1_reviewed = []
for feature in GROUP_5_1_COLUMNS:
    feature_audit = audit_5_1_reviewed.loc[
        audit_5_1_reviewed["feature"].eq(feature)
    ]
    informative_n = int(
        feature_audit.loc[
            feature_audit["status"].eq("valid_informative"),
            "n",
        ].sum()
    )
    special_n = int(
        feature_audit.loc[
            feature_audit["status"].eq("valid_special"),
            "n",
        ].sum()
    )
    blank_or_missing_n = int(
        feature_audit.loc[
            feature_audit["status"].eq("blank_or_missing"),
            "n",
        ].sum()
    )
    invalid_code_n = int(
        feature_audit.loc[
            feature_audit["status"].eq("invalid_code"),
            "n",
        ].sum()
    )
    valid_n = informative_n + special_n

    quality_rows_5_1_reviewed.append({
        "feature": feature,
        "row_count": len(group_5_1_data),
        "valid_n": valid_n,
        "valid_pct": round(valid_n / len(group_5_1_data) * 100, 4),
        "informative_n": informative_n,
        "informative_pct": round(
            informative_n / len(group_5_1_data) * 100,
            4,
        ),
        "special_n": special_n,
        "special_pct": round(
            special_n / len(group_5_1_data) * 100,
            4,
        ),
        "blank_or_missing_n": blank_or_missing_n,
        "invalid_code_n": invalid_code_n,
        "raw_unique_count": group_5_1_data[feature].nunique(
            dropna=False
        ),
    })

quality_5_1_reviewed = pd.DataFrame(quality_rows_5_1_reviewed)
invalid_codes_5_1_reviewed = audit_5_1_reviewed.loc[
    audit_5_1_reviewed["status"].eq("invalid_code")
].copy()

validation_5_1_reviewed = pd.Series({
    "row_count_reconciled": quality_5_1_reviewed[
        "row_count"
    ].eq(len(dataset_b)).all(),
    "audit_counts_reconciled": audit_5_1_reviewed.groupby(
        "feature"
    )["n"].sum().eq(len(dataset_b)).all(),
    "schema_map_matches_observed_tokens": (
        invalid_codes_5_1_reviewed.empty
    ),
})

assert validation_5_1_reviewed.all()
display(quality_5_1_reviewed)
display(audit_5_1_reviewed)
display(validation_5_1_reviewed)

,feature,row_count,valid_n,valid_pct,informative_n,informative_pct,special_n,special_pct,blank_or_missing_n,invalid_code_n,raw_unique_count
0,preapproval,293301,293301,100.0,293301,100.0000,0,0.0000,0,0,2
1,submission_of_application,293301,293301,100.0,285027,97.1790,8274,2.8210,0,0,3
2,initially_payable_to_institution,293301,293301,100.0,279494,95.2925,13807,4.7075,0,0,4
3,conforming_loan_limit,293301,293301,100.0,290136,98.9209,3165,1.0791,0,0,4


,feature,raw_code,description,status,n,share_pct
0,preapproval,2,Preapproval not requested,valid_informative,283785,96.7556
1,preapproval,1,Preapproval requested,valid_informative,9516,3.2444
2,submission_of_application,1,Submitted directly to reporting institution,valid_informative,245493,83.7000
3,submission_of_application,2,Not submitted directly to reporting institution,valid_informative,39534,13.4790
4,submission_of_application,1111,Exempt,valid_special,8274,2.8210
5,initially_payable_to_institution,1,Initially payable to reporting institution,valid_informative,269984,92.0501
6,initially_payable_to_institution,2,Not initially payable to reporting institution,valid_informative,9510,3.2424
7,initially_payable_to_institution,1111,Exempt,valid_special,8274,2.8210
8,initially_payable_to_institution,3,Not applicable,valid_special,5533,1.8865
9,conforming_loan_limit,C,Conforming,valid_informative,280700,95.7037


row_count_reconciled                  True
audit_counts_reconciled               True
schema_map_matches_observed_tokens    True
dtype: bool

In [14]:
submission_informative_codes_5_1 = {"1", "2"}
initial_payable_informative_codes_5_1 = {"1", "2"}

informative_pair_mask_5_1 = (
    group_5_1_data["submission_of_application"].isin(
        submission_informative_codes_5_1
    )
    & group_5_1_data["initially_payable_to_institution"].isin(
        initial_payable_informative_codes_5_1
    )
)

informative_pair_data_5_1 = group_5_1_data.loc[
    informative_pair_mask_5_1
].copy()

informative_pair_counts_5_1, informative_pair_row_pct_5_1 = (
    row_percentage_crosstab_5_1(
        informative_pair_data_5_1,
        "submission_of_application",
        "initially_payable_to_institution",
    )
)

full_pair_cramers_v_5_1 = cramers_v_5_1(
    group_5_1_data,
    "submission_of_application",
    "initially_payable_to_institution",
)
informative_pair_cramers_v_5_1 = cramers_v_5_1(
    informative_pair_data_5_1,
    "submission_of_application",
    "initially_payable_to_institution",
)

association_population_comparison_5_1 = pd.DataFrame([
    {
        "population": "full_cohort",
        "n": len(group_5_1_data),
        "cramers_v": round(full_pair_cramers_v_5_1, 4),
    },
    {
        "population": "informative_only_codes_1_2",
        "n": len(informative_pair_data_5_1),
        "cramers_v": round(informative_pair_cramers_v_5_1, 4),
    },
])

preapproval_requested_mask_5_1 = group_5_1_data[
    "preapproval"
].eq("1")
home_purchase_mask_5_1 = group_5_1_data["loan_purpose"].eq("1")
requested_preapproval_n_5_1 = int(preapproval_requested_mask_5_1.sum())
home_purchase_n_5_1 = int(home_purchase_mask_5_1.sum())
requested_home_purchase_n_5_1 = int(
    (preapproval_requested_mask_5_1 & home_purchase_mask_5_1).sum()
)

preapproval_applicability_5_1 = pd.Series({
    "requested_preapproval_n": requested_preapproval_n_5_1,
    "requested_home_purchase_n": requested_home_purchase_n_5_1,
    "requested_within_home_purchase_pct": round(
        requested_home_purchase_n_5_1
        / requested_preapproval_n_5_1
        * 100,
        4,
    ),
    "home_purchase_n": home_purchase_n_5_1,
    "requested_share_of_home_purchase_pct": round(
        requested_home_purchase_n_5_1
        / home_purchase_n_5_1
        * 100,
        4,
    ),
})

assert len(informative_pair_data_5_1) > 0
assert requested_preapproval_n_5_1 > 0
assert requested_home_purchase_n_5_1 == requested_preapproval_n_5_1

display(association_population_comparison_5_1)
display(informative_pair_counts_5_1)
display(informative_pair_row_pct_5_1)
display(preapproval_applicability_5_1)

,population,n,cramers_v
0,full_cohort,293301,0.7178
1,informative_only_codes_1_2,279494,0.1735


initially_payable_to_institution,1,2
submission_of_application,,
1,236015,5184
2,33969,4326


initially_payable_to_institution,1,2
submission_of_application,,
1,97.85,2.15
2,88.70,11.30


requested_preapproval_n                   9516.000
requested_home_purchase_n                 9516.000
requested_within_home_purchase_pct         100.000
home_purchase_n                         139144.000
requested_share_of_home_purchase_pct         6.839
dtype: float64

In [15]:
global_dominance_rows_5_1 = []
for feature in GROUP_5_1_COLUMNS:
    feature_counts = group_5_1_data[feature].value_counts(dropna=False)
    global_dominance_rows_5_1.append({
        "feature": feature,
        "global_dominant_share_pct": round(
            feature_counts.iloc[0] / len(group_5_1_data) * 100,
            4,
        ),
    })

global_dominance_5_1 = pd.DataFrame(global_dominance_rows_5_1)
lei_proxy_enhanced_5_1 = lei_proxy_summary_5_1.merge(
    global_dominance_5_1,
    on="feature",
    validate="one_to_one",
)
lei_proxy_enhanced_5_1["lei_uplift_vs_global_pp"] = (
    lei_proxy_enhanced_5_1["row_weighted_dominant_share_pct"]
    - lei_proxy_enhanced_5_1["global_dominant_share_pct"]
).round(4)

lei_size_profile_5_1 = (
    lei_proxy_detail_5_1.loc[
        lei_proxy_detail_5_1["feature"].eq(GROUP_5_1_COLUMNS[0]),
        ["lei", "row_count"],
    ]
    .drop_duplicates("lei")
    .sort_values(["row_count", "lei"])
    .reset_index(drop=True)
)
lei_size_profile_5_1["lei_size_quartile"] = pd.qcut(
    lei_size_profile_5_1["row_count"].rank(method="first"),
    q=4,
    labels=["Q1_smallest", "Q2", "Q3", "Q4_largest"],
)

lei_proxy_detail_with_size_5_1 = lei_proxy_detail_5_1.merge(
    lei_size_profile_5_1[["lei", "lei_size_quartile"]],
    on="lei",
    validate="many_to_one",
)

lei_size_sensitivity_rows_5_1 = []
for feature in GROUP_5_1_COLUMNS:
    feature_detail = lei_proxy_detail_with_size_5_1.loc[
        lei_proxy_detail_with_size_5_1["feature"].eq(feature)
    ]
    for size_quartile, size_detail in feature_detail.groupby(
        "lei_size_quartile",
        observed=True,
    ):
        lei_size_sensitivity_rows_5_1.append({
            "feature": feature,
            "lei_size_quartile": str(size_quartile),
            "lei_n": size_detail["lei"].nunique(),
            "row_n": int(size_detail["row_count"].sum()),
            "row_weighted_dominant_share_pct": round(
                np.average(
                    size_detail["dominant_share"],
                    weights=size_detail["row_count"],
                )
                * 100,
                4,
            ),
        })

lei_size_sensitivity_5_1 = pd.DataFrame(
    lei_size_sensitivity_rows_5_1
)
lei_size_distribution_5_1 = lei_size_profile_5_1[
    "row_count"
].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

assert lei_proxy_enhanced_5_1["lei_uplift_vs_global_pp"].notna().all()
assert lei_size_sensitivity_5_1["row_n"].gt(0).all()

display(lei_proxy_enhanced_5_1)
display(lei_size_distribution_5_1)
display(lei_size_sensitivity_5_1)

,feature,scope,lei_n,blank_lei_row_n,single_category_lei_n,single_category_lei_pct,median_dominant_share_pct,row_weighted_dominant_share_pct,global_dominant_share_pct,lei_uplift_vs_global_pp
0,preapproval,full_cohort_descriptive_only,715,0,588,82.24,100.0,97.34,96.7556,0.5844
1,submission_of_application,full_cohort_descriptive_only,715,0,603,84.34,100.0,96.30,83.7000,12.6000
2,initially_payable_to_institution,full_cohort_descriptive_only,715,0,603,84.34,100.0,96.57,92.0501,4.5199
3,conforming_loan_limit,full_cohort_descriptive_only,715,0,377,52.73,100.0,95.93,95.7037,0.2263


count      715.000000
mean       410.211189
std       1392.652345
min          1.000000
25%          2.000000
50%         25.000000
75%        195.500000
90%        893.400000
95%       1818.500000
99%       7554.100000
max      14757.000000
Name: row_count, dtype: float64

,feature,lei_size_quartile,lei_n,row_n,row_weighted_dominant_share_pct
0,preapproval,Q1_smallest,179,226,100.0000
1,preapproval,Q2,179,1715,97.7259
2,preapproval,Q3,178,15683,96.0148
3,preapproval,Q4_largest,179,275677,97.4151
4,submission_of_application,Q1_smallest,179,226,98.6726
5,submission_of_application,Q2,179,1715,97.4927
6,submission_of_application,Q3,178,15683,96.3910
7,submission_of_application,Q4_largest,179,275677,96.2830
8,initially_payable_to_institution,Q1_smallest,179,226,98.6726
9,initially_payable_to_institution,Q2,179,1715,97.9009


In [16]:
quality_lookup_5_1_reviewed = quality_5_1_reviewed.set_index("feature")
lei_lookup_5_1_reviewed = lei_proxy_enhanced_5_1.set_index("feature")

timing_evidence_5_1 = pd.DataFrame([
    {
        "feature": "preapproval",
        "timing_status": "available_before_decision_by_definition",
        "timing_evidence": (
            "2024 HMDA FIG 4(a)(4): whether the application involved "
            "a request for preapproval."
        ),
    },
    {
        "feature": "submission_of_application",
        "timing_status": "available_before_decision_by_definition",
        "timing_evidence": (
            "2024 HMDA FIG 4(a)(33): whether the applicant submitted "
            "the application directly to the institution."
        ),
    },
    {
        "feature": "initially_payable_to_institution",
        "timing_status": "available_before_decision_by_definition",
        "timing_evidence": (
            "2024 HMDA FIG 4(a)(33): for an application, whether the "
            "obligation would have been initially payable to the institution."
        ),
    },
    {
        "feature": "conforming_loan_limit",
        "timing_status": "not_consistently_predecision_safe",
        "timing_evidence": (
            "CFPB Regulation C timing: reported loan amount uses the legal-obligation "
            "amount for originated loans, approved amount for action 2, and amount "
            "applied for for denied loans; this public-derived flag is excluded."
        ),
    },
])

screening_decision_5_1 = pd.DataFrame({
    "feature_or_block": GROUP_5_1_COLUMNS,
}).merge(
    timing_evidence_5_1,
    left_on="feature_or_block",
    right_on="feature",
    validate="one_to_one",
).drop(columns="feature")

screening_decision_5_1["quality_evidence"] = (
    screening_decision_5_1["feature_or_block"].map({
        feature: (
            f"{quality_lookup_5_1_reviewed.loc[feature, 'valid_pct']:.2f}% "
            f"valid; "
            f"{quality_lookup_5_1_reviewed.loc[feature, 'informative_pct']:.2f}% "
            f"informative; "
            f"{int(quality_lookup_5_1_reviewed.loc[feature, 'invalid_code_n'])} "
            "invalid codes."
        )
        for feature in GROUP_5_1_COLUMNS
    })
)

screening_decision_5_1["conditional_applicability"] = (
    screening_decision_5_1["feature_or_block"].map({
        "preapproval": (
            f"All {requested_preapproval_n_5_1} requested-preapproval rows "
            f"are home purchase; they represent "
            f"{preapproval_applicability_5_1['requested_share_of_home_purchase_pct']:.2f}% "
            "of home-purchase rows."
        ),
        "submission_of_application": "Codes 3 and 1111 retained separately.",
        "initially_payable_to_institution": (
            "Codes 3 and 1111 retained separately."
        ),
        "conforming_loan_limit": "U and NA retained as separate valid special states.",
    })
)

screening_decision_5_1["redundancy_evidence"] = (
    screening_decision_5_1["feature_or_block"].map({
        "preapproval": (
            "Conditional on home-purchase scope; not a duplicate of loan_purpose."
        ),
        "submission_of_application": (
            f"Joint block with initially_payable: full V={full_pair_cramers_v_5_1:.4f}; "
            f"informative-only V={informative_pair_cramers_v_5_1:.4f}."
        ),
        "initially_payable_to_institution": (
            f"Joint block with submission: full V={full_pair_cramers_v_5_1:.4f}; "
            f"informative-only V={informative_pair_cramers_v_5_1:.4f}."
        ),
        "conforming_loan_limit": (
            "Public-derived from outcome-dependent reported loan amount; excluded before validation."
        ),
    })
)

screening_decision_5_1["organization_proxy_evidence"] = (
    screening_decision_5_1["feature_or_block"].map({
        feature: (
            f"LEI row-weighted dominance "
            f"{lei_lookup_5_1_reviewed.loc[feature, 'row_weighted_dominant_share_pct']:.2f}%; "
            f"global dominance "
            f"{lei_lookup_5_1_reviewed.loc[feature, 'global_dominant_share_pct']:.2f}%; "
            f"uplift "
            f"{lei_lookup_5_1_reviewed.loc[feature, 'lei_uplift_vs_global_pp']:+.2f} pp."
        )
        for feature in GROUP_5_1_COLUMNS
    })
)

screening_decision_5_1["protected_group_sensitivity"] = (
    screening_decision_5_1["feature_or_block"].map({
        "conforming_loan_limit": "not_applicable_excluded_before_modeling",
    }).fillna("reserved_for_grouped_train_validation")
)
screening_decision_5_1["screening_status"] = (
    screening_decision_5_1["feature_or_block"].map({
        "preapproval": "advance_to_grouped_validation",
        "submission_of_application": (
            "advance_to_grouped_validation_as_joint_block"
        ),
        "initially_payable_to_institution": (
            "advance_to_grouped_validation_as_joint_block"
        ),
        "conforming_loan_limit": "exclude_from_further_review",
    })
)
screening_decision_5_1["decision"] = pd.NA
screening_decision_5_1["decision_reason"] = (
    "Final keep/exclude is reserved for grouped train/validation evidence "
    "after sections 5.1-5.5 are screened."
)
screening_decision_5_1["gsv_player"] = pd.NA
conforming_row_mask_5_1 = screening_decision_5_1["feature_or_block"].eq(
    "conforming_loan_limit"
)
screening_decision_5_1.loc[conforming_row_mask_5_1, "decision"] = "exclude"
screening_decision_5_1.loc[conforming_row_mask_5_1, "decision_reason"] = (
    "Excluded because the reported loan-amount basis differs by action outcome; "
    "retain only in source/audit."
)
screening_decision_5_1.loc[conforming_row_mask_5_1, "gsv_player"] = "not_applicable"
timing_resolution_mask_5_1 = (
    screening_decision_5_1["timing_status"].eq(
        "available_before_decision_by_definition"
    )
    | (
        screening_decision_5_1["timing_status"].eq(
            "not_consistently_predecision_safe"
        )
        & screening_decision_5_1["screening_status"].eq(
            "exclude_from_further_review"
        )
    )
)

section_5_1_gate = pd.Series({
    "schema_gate": bool(validation_5_1_reviewed[
        "schema_map_matches_observed_tokens"
    ]),
    "informative_redundancy_gate": pd.notna(
        informative_pair_cramers_v_5_1
    ),
    "conditional_applicability_gate": (
        requested_home_purchase_n_5_1 == requested_preapproval_n_5_1
    ),
    "lei_diagnostic_gate": lei_proxy_enhanced_5_1[
        "lei_uplift_vs_global_pp"
    ].notna().all(),
    "screening_record_gate": screening_decision_5_1[
        "screening_status"
    ].notna().all(),
    "timing_gate": timing_resolution_mask_5_1.all(),
})
section_5_1_gate["ready_to_close_5_1"] = section_5_1_gate.all()
section_5_1_gate["ready_to_start_5_2"] = section_5_1_gate[
    "ready_to_close_5_1"
]

assert screening_decision_5_1["screening_status"].notna().all()
assert screening_decision_5_1.loc[
    ~conforming_row_mask_5_1, "decision"
].isna().all()
assert screening_decision_5_1.loc[
    ~conforming_row_mask_5_1, "gsv_player"
].isna().all()
assert screening_decision_5_1.loc[
    conforming_row_mask_5_1, "decision"
].eq("exclude").all()
assert screening_decision_5_1.loc[
    conforming_row_mask_5_1, "gsv_player"
].eq("not_applicable").all()
assert section_5_1_gate["ready_to_start_5_2"]

display(screening_decision_5_1)
display(section_5_1_gate)

,feature_or_block,timing_status,timing_evidence,quality_evidence,conditional_applicability,redundancy_evidence,organization_proxy_evidence,protected_group_sensitivity,screening_status,decision,decision_reason,gsv_player
0,preapproval,available_before_decision_by_definition,2024 HMDA FIG 4(a)(4): whether the application...,100.00% valid; 100.00% informative; 0 invalid ...,All 9516 requested-preapproval rows are home p...,Conditional on home-purchase scope; not a dupl...,LEI row-weighted dominance 97.34%; global domi...,reserved_for_grouped_train_validation,advance_to_grouped_validation,<NA>,Final keep/exclude is reserved for grouped tra...,<NA>
1,submission_of_application,available_before_decision_by_definition,2024 HMDA FIG 4(a)(33): whether the applicant ...,100.00% valid; 97.18% informative; 0 invalid c...,Codes 3 and 1111 retained separately.,Joint block with initially_payable: full V=0.7...,LEI row-weighted dominance 96.30%; global domi...,reserved_for_grouped_train_validation,advance_to_grouped_validation_as_joint_block,<NA>,Final keep/exclude is reserved for grouped tra...,<NA>
2,initially_payable_to_institution,available_before_decision_by_definition,"2024 HMDA FIG 4(a)(33): for an application, wh...",100.00% valid; 95.29% informative; 0 invalid c...,Codes 3 and 1111 retained separately.,Joint block with submission: full V=0.7178; in...,LEI row-weighted dominance 96.57%; global domi...,reserved_for_grouped_train_validation,advance_to_grouped_validation_as_joint_block,<NA>,Final keep/exclude is reserved for grouped tra...,<NA>
3,conforming_loan_limit,not_consistently_predecision_safe,CFPB Regulation C timing: reported loan amount...,100.00% valid; 98.92% informative; 0 invalid c...,U and NA retained as separate valid special st...,Public-derived from outcome-dependent reported...,LEI row-weighted dominance 95.93%; global domi...,not_applicable_excluded_before_modeling,exclude_from_further_review,exclude,Excluded because the reported loan-amount basi...,not_applicable


schema_gate                       True
informative_redundancy_gate       True
conditional_applicability_gate    True
lei_diagnostic_gate               True
screening_record_gate             True
timing_gate                       True
ready_to_close_5_1                True
ready_to_start_5_2                True
dtype: bool

### 5.1.2. Kết luận screening

**Observed facts**

- `NA` của `conforming_loan_limit` là valid special state (*Not Applicable*), không phải invalid code. Sau khi bổ sung mapping, bốn feature có valid-code coverage 100%.
- `submission_of_application` và `initially_payable_to_institution` có Cramér's V bằng 0,7178 trên full cohort nhưng khoảng 0,1744 khi chỉ giữ code informative `1`/`2`. Cặp `1111` khớp hoàn toàn là nguyên nhân quan trọng làm association toàn cohort tăng mạnh.
- Toàn bộ 9.516 hồ sơ có requested preapproval thuộc home purchase; chúng chiếm khoảng 6,84% số hồ sơ home purchase. Đây là conditional applicability, không phải bằng chứng `preapproval` là bản sao của `loan_purpose`.
- LEI uplift so với global dominance lớn nhất ở `submission_of_application` (khoảng +12,60 điểm phần trăm), tiếp theo là `initially_payable_to_institution` (khoảng +4,52); `preapproval` và `conforming_loan_limit` chỉ tăng nhẹ. LEI chỉ là diagnostic guardrail và không được đưa vào `X`.

**Interpretation**

- `preapproval` có thể đi tiếp nhưng phải được đánh giá trong ngữ cảnh home purchase.
- Hai trường application-channel đi tiếp như một joint business block; không loại một trường chỉ bằng ngưỡng Cramér's V.
- `conforming_loan_limit` đạt schema gate nhưng bị loại khỏi modeling: CFPB Regulation C đo reported loan amount khác theo action outcome, nên public-derived flag này không nhất quán pre-decision-safe.

**Screening outcome**

- `preapproval`: `advance_to_grouped_validation`.
- `submission_of_application` + `initially_payable_to_institution`: `advance_to_grouped_validation_as_joint_block`.
- `conforming_loan_limit`: `exclude_from_further_review`; `decision = exclude`; `gsv_player = not_applicable`; field vẫn giữ trong source/audit.
- `decision` và `gsv_player` của ba feature đi tiếp vẫn để trống; chỉ điền sau grouped train/validation, stability và fairness sensitivity theo protocol.

Timing risk đã được xử lý bằng exclusion có bằng chứng. Vì vậy phần 5.1 **đủ điều kiện đóng ở mức screening và có thể bắt đầu 5.2**. Không có kết luận về model performance, feature importance hoặc quan hệ nhân quả trong phần này.

## 5.2. Cấu trúc trả nợ đặc biệt

Screening này đánh giá bốn repayment flags về schema, timing, coverage, overlap, LEI guardrail và protected-metadata sensitivity. Không fit model, không mở test set và không quyết định final keep/exclude. Theo 2024 HMDA FIG 4(a)(27), code `1`, `2` và `1111` được giữ riêng; `1111` là reporting exemption, không phải "No".


In [17]:
GROUP_5_2_COLUMNS = [
    "negative_amortization",
    "interest_only_payment",
    "balloon_payment",
    "other_nonamortizing_features",
]
RACE_SLOT_COLUMNS_5_2 = [f"applicant_race-{slot}" for slot in range(1, 6)]
REFERENCE_5_2_COLUMNS = ["lei", "applicant_sex"] + RACE_SLOT_COLUMNS_5_2
required_columns_5_2 = GROUP_5_2_COLUMNS + REFERENCE_5_2_COLUMNS
missing_columns_5_2 = [
    column for column in required_columns_5_2
    if column not in dataset_b.columns
]
assert not missing_columns_5_2, f"Missing columns: {missing_columns_5_2}"
group_5_2_data = dataset_b.loc[:, required_columns_5_2].copy()

EXPECTED_CODE_LABELS_5_2 = {
    "negative_amortization": {
        "1": "Negative amortization",
        "2": "No negative amortization",
        "1111": "Exempt",
    },
    "interest_only_payment": {
        "1": "Interest-only payments",
        "2": "No interest-only payments",
        "1111": "Exempt",
    },
    "balloon_payment": {
        "1": "Balloon payment",
        "2": "No balloon payment",
        "1111": "Exempt",
    },
    "other_nonamortizing_features": {
        "1": "Other non-fully amortizing features",
        "2": "No other non-amortizing features",
        "1111": "Exempt",
    },
}
SPECIAL_CODES_5_2 = {feature: {"1111"} for feature in GROUP_5_2_COLUMNS}
INFORMATIVE_CODES_5_2 = {"1", "2"}

def audit_feature_codes_5_2(data, feature):
    rows = []
    for raw_code, count in data[feature].value_counts(dropna=False).items():
        if pd.isna(raw_code) or str(raw_code).strip() == "":
            status, description = "blank_or_missing", "Blank or missing"
        elif raw_code not in EXPECTED_CODE_LABELS_5_2[feature]:
            status, description = "invalid_code", "Unrecognized code"
        elif raw_code in SPECIAL_CODES_5_2[feature]:
            status, description = "valid_special", EXPECTED_CODE_LABELS_5_2[feature][raw_code]
        else:
            status, description = "valid_informative", EXPECTED_CODE_LABELS_5_2[feature][raw_code]
        rows.append({
            "feature": feature,
            "raw_code": raw_code,
            "description": description,
            "status": status,
            "n": int(count),
            "share_pct": round(count / len(data) * 100, 4),
        })
    return pd.DataFrame(rows)

audit_5_2 = pd.concat(
    [audit_feature_codes_5_2(group_5_2_data, feature) for feature in GROUP_5_2_COLUMNS],
    ignore_index=True,
)
quality_rows_5_2 = []
for feature in GROUP_5_2_COLUMNS:
    counts = (
        audit_5_2.loc[audit_5_2["feature"].eq(feature)]
        .groupby("status")["n"]
        .sum()
    )
    informative_n = int(counts.get("valid_informative", 0))
    special_n = int(counts.get("valid_special", 0))
    positive_n = int(group_5_2_data[feature].eq("1").sum())
    quality_rows_5_2.append({
        "feature": feature,
        "row_count": len(group_5_2_data),
        "valid_n": informative_n + special_n,
        "valid_pct": round((informative_n + special_n) / len(group_5_2_data) * 100, 4),
        "informative_n": informative_n,
        "informative_pct": round(informative_n / len(group_5_2_data) * 100, 4),
        "positive_n": positive_n,
        "positive_pct": round(positive_n / len(group_5_2_data) * 100, 4),
        "special_n": special_n,
        "special_pct": round(special_n / len(group_5_2_data) * 100, 4),
        "blank_or_missing_n": int(counts.get("blank_or_missing", 0)),
        "invalid_code_n": int(counts.get("invalid_code", 0)),
    })
quality_5_2 = pd.DataFrame(quality_rows_5_2)
invalid_codes_5_2 = audit_5_2.loc[audit_5_2["status"].eq("invalid_code")].copy()
quality_validation_5_2 = pd.Series({
    "row_count_reconciled": quality_5_2["row_count"].eq(len(dataset_b)).all(),
    "audit_counts_reconciled": audit_5_2.groupby("feature")["n"].sum().eq(len(dataset_b)).all(),
    "schema_map_matches_observed_tokens": invalid_codes_5_2.empty,
})
assert quality_validation_5_2.all()
display(quality_5_2)
display(audit_5_2)
display(quality_validation_5_2)


,feature,row_count,valid_n,valid_pct,informative_n,informative_pct,positive_n,positive_pct,special_n,special_pct,blank_or_missing_n,invalid_code_n
0,negative_amortization,293301,293301,100.0,284657,97.0529,401,0.1367,8644,2.9471,0,0
1,interest_only_payment,293301,293301,100.0,284657,97.0529,56571,19.2877,8644,2.9471,0,0
2,balloon_payment,293301,293301,100.0,284657,97.0529,6965,2.3747,8644,2.9471,0,0
3,other_nonamortizing_features,293301,293301,100.0,284657,97.0529,411,0.1401,8644,2.9471,0,0


,feature,raw_code,description,status,n,share_pct
0,negative_amortization,2,No negative amortization,valid_informative,284256,96.9161
1,negative_amortization,1111,Exempt,valid_special,8644,2.9471
2,negative_amortization,1,Negative amortization,valid_informative,401,0.1367
3,interest_only_payment,2,No interest-only payments,valid_informative,228086,77.7652
4,interest_only_payment,1,Interest-only payments,valid_informative,56571,19.2877
5,interest_only_payment,1111,Exempt,valid_special,8644,2.9471
6,balloon_payment,2,No balloon payment,valid_informative,277692,94.6782
7,balloon_payment,1111,Exempt,valid_special,8644,2.9471
8,balloon_payment,1,Balloon payment,valid_informative,6965,2.3747
9,other_nonamortizing_features,2,No other non-amortizing features,valid_informative,284246,96.9127


row_count_reconciled                  True
audit_counts_reconciled               True
schema_map_matches_observed_tokens    True
dtype: bool

In [18]:
def cramers_v_5_2(data, left, right):
    observed = pd.crosstab(data[left], data[right], dropna=False).to_numpy(dtype=float)
    n = observed.sum()
    if n == 0 or min(observed.shape) < 2:
        return np.nan
    expected = np.outer(observed.sum(axis=1), observed.sum(axis=0)) / n
    chi_square = np.divide(
        (observed - expected) ** 2,
        expected,
        out=np.zeros_like(expected),
        where=expected > 0,
    ).sum()
    return float(
        np.sqrt(
            (chi_square / n)
            / min(observed.shape[0] - 1, observed.shape[1] - 1)
        )
    )

informative_mask_5_2 = group_5_2_data[GROUP_5_2_COLUMNS].isin(
    INFORMATIVE_CODES_5_2
).all(axis=1)
informative_data_5_2 = group_5_2_data.loc[informative_mask_5_2].copy()
association_rows_5_2 = []
for left, right in combinations(GROUP_5_2_COLUMNS, 2):
    for population, data in [
        ("full_cohort", group_5_2_data),
        ("informative_only", informative_data_5_2),
    ]:
        association_rows_5_2.append({
            "left_feature": left,
            "right_feature": right,
            "population": population,
            "n": len(data),
            "cramers_v": round(cramers_v_5_2(data, left, right), 4),
        })
association_5_2 = pd.DataFrame(association_rows_5_2)
association_by_feature_5_2 = pd.concat([
    association_5_2.rename(columns={"left_feature": "feature"})[
        ["feature", "population", "cramers_v"]
    ],
    association_5_2.rename(columns={"right_feature": "feature"})[
        ["feature", "population", "cramers_v"]
    ],
]).groupby(["feature", "population"])["cramers_v"].max().unstack("population")

repayment_patterns_5_2 = (
    group_5_2_data.groupby(GROUP_5_2_COLUMNS, dropna=False)
    .size()
    .rename("n")
    .reset_index()
    .sort_values("n", ascending=False)
    .reset_index(drop=True)
)
repayment_patterns_5_2["share_pct"] = (
    repayment_patterns_5_2["n"] / len(group_5_2_data) * 100
).round(4)
any_special_repayment_5_2 = group_5_2_data[GROUP_5_2_COLUMNS].eq("1").any(axis=1)
block_prevalence_5_2 = pd.Series({
    "full_cohort_n": len(group_5_2_data),
    "informative_all_four_n": int(informative_mask_5_2.sum()),
    "any_special_repayment_n": int(any_special_repayment_5_2.sum()),
    "any_special_repayment_pct": round(any_special_repayment_5_2.mean() * 100, 4),
})
assert repayment_patterns_5_2["n"].sum() == len(group_5_2_data)
assert len(informative_data_5_2) > 0
display(block_prevalence_5_2)
display(association_5_2)
display(repayment_patterns_5_2)


full_cohort_n                293301.0000
informative_all_four_n       284657.0000
any_special_repayment_n       59655.0000
any_special_repayment_pct        20.3392
dtype: float64

,left_feature,right_feature,population,n,cramers_v
0,negative_amortization,interest_only_payment,full_cohort,293301,0.7072
1,negative_amortization,interest_only_payment,informative_only,284657,0.0168
2,negative_amortization,balloon_payment,full_cohort,293301,0.7071
3,negative_amortization,balloon_payment,informative_only,284657,0.0017
4,negative_amortization,other_nonamortizing_features,full_cohort,293301,0.7071
5,negative_amortization,other_nonamortizing_features,informative_only,284657,0.0010
6,interest_only_payment,balloon_payment,full_cohort,293301,0.7173
7,interest_only_payment,balloon_payment,informative_only,284657,0.1701
8,interest_only_payment,other_nonamortizing_features,full_cohort,293301,0.7075
9,interest_only_payment,other_nonamortizing_features,informative_only,284657,0.0337


,negative_amortization,interest_only_payment,balloon_payment,other_nonamortizing_features,n,share_pct
0,2,2,2,2,225002,76.7137
1,2,1,2,2,51986,17.7245
2,1111,1111,1111,1111,8644,2.9471
3,2,1,1,2,4351,1.4835
4,2,2,1,2,2507,0.8548
5,1,2,2,2,393,0.1340
6,2,1,2,1,214,0.0730
7,2,2,2,1,96,0.0327
8,2,2,1,1,88,0.0300
9,2,1,1,1,12,0.0041


In [19]:
valid_lei_mask_5_2 = (
    group_5_2_data["lei"].notna()
    & group_5_2_data["lei"].str.strip().ne("")
)
lei_data_5_2 = group_5_2_data.loc[
    valid_lei_mask_5_2,
    ["lei"] + GROUP_5_2_COLUMNS,
].copy()
lei_rows_5_2 = []
for feature in GROUP_5_2_COLUMNS:
    counts = (
        lei_data_5_2.groupby(["lei", feature], dropna=False)
        .size()
        .rename("n")
        .reset_index()
    )
    counts["lei_total_n"] = counts.groupby("lei")["n"].transform("sum")
    per_lei = (
        counts.assign(within_lei_share=counts["n"] / counts["lei_total_n"])
        .groupby("lei")
        .agg(
            row_count=("n", "sum"),
            dominant_share=("within_lei_share", "max"),
        )
        .reset_index()
    )
    positive_by_lei = (
        lei_data_5_2.assign(is_positive=lei_data_5_2[feature].eq("1"))
        .groupby("lei")["is_positive"]
        .sum()
    )
    positive_n = int(positive_by_lei.sum())
    positive_lei_n = int(positive_by_lei.gt(0).sum())
    global_dominance = (
        lei_data_5_2[feature].value_counts(dropna=False).iloc[0]
        / len(lei_data_5_2)
        * 100
    )
    weighted_dominance = (
        np.average(per_lei["dominant_share"], weights=per_lei["row_count"])
        * 100
    )
    lei_rows_5_2.append({
        "feature": feature,
        "lei_n": per_lei["lei"].nunique(),
        "global_dominant_share_pct": round(global_dominance, 4),
        "row_weighted_dominant_share_pct": round(weighted_dominance, 4),
        "lei_uplift_vs_global_pp": round(weighted_dominance - global_dominance, 4),
        "positive_lei_n": positive_lei_n,
        "positive_top_1_lei_share_pct": (
            0.0 if positive_n == 0 else round(positive_by_lei.nlargest(1).sum() / positive_n * 100, 4)
        ),
        "positive_top_10_lei_share_pct": (
            0.0
            if positive_n == 0
            else round(positive_by_lei.nlargest(10).sum() / positive_n * 100, 4)
        ),
    })
lei_proxy_5_2 = pd.DataFrame(lei_rows_5_2)

def canonical_race_token_set_5_2(row):
    tokens = sorted({str(value).strip() for value in row if str(value).strip()})
    return "|".join(tokens) if tokens else "blank"

protected_data_5_2 = group_5_2_data[
    ["applicant_sex"] + RACE_SLOT_COLUMNS_5_2 + GROUP_5_2_COLUMNS
].copy()
protected_data_5_2["applicant_race_token_set"] = protected_data_5_2[
    RACE_SLOT_COLUMNS_5_2
].apply(canonical_race_token_set_5_2, axis=1)
protected_rows_5_2 = []
for protected_group in ["applicant_sex", "applicant_race_token_set"]:
    for feature in GROUP_5_2_COLUMNS:
        summary = (
            protected_data_5_2.groupby(protected_group, dropna=False)[feature]
            .agg(
                n="size",
                positive_n=lambda values: values.eq("1").sum(),
                special_n=lambda values: values.eq("1111").sum(),
            )
            .reset_index()
        )
        summary["protected_group"] = protected_group
        summary["feature"] = feature
        summary["positive_pct"] = (summary["positive_n"] / summary["n"] * 100).round(4)
        summary["special_pct"] = (summary["special_n"] / summary["n"] * 100).round(4)
        protected_rows_5_2.append(summary)
protected_sensitivity_5_2 = pd.concat(protected_rows_5_2, ignore_index=True)
assert lei_proxy_5_2["lei_uplift_vs_global_pp"].notna().all()
assert not protected_sensitivity_5_2.empty
display(lei_proxy_5_2)
display(protected_sensitivity_5_2.sort_values("n", ascending=False).head(40))

quality_lookup_5_2 = quality_5_2.set_index("feature")
lei_lookup_5_2 = lei_proxy_5_2.set_index("feature")
screening_decision_5_2 = pd.DataFrame({"feature_or_block": GROUP_5_2_COLUMNS})
screening_decision_5_2["timing_status"] = "available_before_decision_by_definition"
screening_decision_5_2["timing_evidence"] = (
    "2024 HMDA FIG 4(a)(27): contractual terms include, or would have included, "
    "the repayment feature for a covered loan or application."
)
screening_decision_5_2["quality_evidence"] = screening_decision_5_2[
    "feature_or_block"
].map({
    feature: (
        f"{quality_lookup_5_2.loc[feature, 'valid_pct']:.2f}% valid; "
        f"{quality_lookup_5_2.loc[feature, 'informative_pct']:.2f}% informative; "
        f"{quality_lookup_5_2.loc[feature, 'positive_pct']:.2f}% code-1; "
        f"{int(quality_lookup_5_2.loc[feature, 'invalid_code_n'])} invalid codes."
    )
    for feature in GROUP_5_2_COLUMNS
})
screening_decision_5_2["conditional_applicability"] = (
    "All-cohort reporting field; code 1111 retained as reporting exemption, not imputed as No."
)
screening_decision_5_2["redundancy_evidence"] = screening_decision_5_2[
    "feature_or_block"
].map({
    feature: (
        f"Maximum pairwise V: full={association_by_feature_5_2.loc[feature, 'full_cohort']:.4f}; "
        f"informative-only={association_by_feature_5_2.loc[feature, 'informative_only']:.4f}."
    )
    for feature in GROUP_5_2_COLUMNS
})
screening_decision_5_2["organization_proxy_evidence"] = screening_decision_5_2[
    "feature_or_block"
].map({
    feature: (
        f"LEI row-weighted dominance {lei_lookup_5_2.loc[feature, 'row_weighted_dominant_share_pct']:.2f}%; "
        f"global {lei_lookup_5_2.loc[feature, 'global_dominant_share_pct']:.2f}%; "
        f"uplift {lei_lookup_5_2.loc[feature, 'lei_uplift_vs_global_pp']:+.2f} pp; "
        f"code-1 occurs in {int(lei_lookup_5_2.loc[feature, 'positive_lei_n'])} LEIs; "
        f"top-1/top-10 LEIs hold {lei_lookup_5_2.loc[feature, 'positive_top_1_lei_share_pct']:.2f}%/"
        f"{lei_lookup_5_2.loc[feature, 'positive_top_10_lei_share_pct']:.2f}% of code-1 rows."
    )
    for feature in GROUP_5_2_COLUMNS
})
screening_decision_5_2["protected_group_sensitivity"] = (
    screening_decision_5_2["feature_or_block"].map({
        "negative_amortization": "not_applicable_excluded_before_modeling",
        "other_nonamortizing_features": "not_applicable_excluded_before_modeling",
    }).fillna("Descriptive sex and canonical race-token-set summaries completed; not a decision rule.")
)
screening_decision_5_2["screening_status"] = (
    screening_decision_5_2["feature_or_block"].map({
        "negative_amortization": "exclude_from_further_review",
        "interest_only_payment": "advance_to_grouped_validation_as_joint_block",
        "balloon_payment": "advance_to_grouped_validation_as_joint_block",
        "other_nonamortizing_features": "exclude_from_further_review",
    })
)
screening_decision_5_2["decision"] = pd.NA
screening_decision_5_2["decision_reason"] = (
    "Final keep/exclude and player mapping are reserved for grouped train/validation "
    "after sections 5.1-5.5."
)
screening_decision_5_2["gsv_player"] = pd.NA
excluded_proxy_mask_5_2 = screening_decision_5_2["feature_or_block"].isin([
    "negative_amortization", "other_nonamortizing_features",
])
screening_decision_5_2.loc[excluded_proxy_mask_5_2, "decision"] = "exclude"
screening_decision_5_2.loc[excluded_proxy_mask_5_2, "decision_reason"] = (
    "Excluded for extreme rarity and concentration of code-1 observations in very few LEIs; retain only in source/audit."
)
screening_decision_5_2.loc[excluded_proxy_mask_5_2, "gsv_player"] = "not_applicable"
section_5_2_gate = pd.Series({
    "prior_5_1_gate": bool(section_5_1_gate["ready_to_start_5_2"]),
    "schema_gate": bool(quality_validation_5_2["schema_map_matches_observed_tokens"]),
    "timing_gate": screening_decision_5_2["timing_status"].eq(
        "available_before_decision_by_definition"
    ).all(),
    "informative_overlap_gate": association_5_2.loc[
        association_5_2["population"].eq("informative_only"), "cramers_v"
    ].notna().all(),
    "lei_diagnostic_gate": lei_proxy_5_2["lei_uplift_vs_global_pp"].notna().all(),
    "protected_sensitivity_gate": not protected_sensitivity_5_2.empty,
    "screening_record_gate": screening_decision_5_2["screening_status"].notna().all(),
})
section_5_2_gate["ready_to_close_5_2"] = section_5_2_gate.all()
section_5_2_gate["ready_to_start_5_3"] = section_5_2_gate["ready_to_close_5_2"]
assert screening_decision_5_2.loc[~excluded_proxy_mask_5_2, "decision"].isna().all()
assert screening_decision_5_2.loc[~excluded_proxy_mask_5_2, "gsv_player"].isna().all()
assert screening_decision_5_2.loc[excluded_proxy_mask_5_2, "decision"].eq("exclude").all()
assert screening_decision_5_2.loc[excluded_proxy_mask_5_2, "gsv_player"].eq("not_applicable").all()
assert section_5_2_gate["ready_to_close_5_2"]
display(screening_decision_5_2)
display(section_5_2_gate)


,feature,lei_n,global_dominant_share_pct,row_weighted_dominant_share_pct,lei_uplift_vs_global_pp,positive_lei_n,positive_top_1_lei_share_pct,positive_top_10_lei_share_pct
0,negative_amortization,715,96.9161,99.9489,3.0327,4,97.7556,100.0000
1,interest_only_payment,715,77.7652,89.3730,11.6079,278,17.8077,70.2127
2,balloon_payment,715,94.6782,98.3392,3.6611,179,19.6267,57.3439
3,other_nonamortizing_features,715,96.9127,99.8285,2.9158,18,53.2847,97.3236


,applicant_sex,n,positive_n,special_n,protected_group,feature,positive_pct,special_pct,applicant_race_token_set
694,<NA>,187507,1031,5990,applicant_race_token_set,balloon_payment,0.5498,3.1945,5
920,<NA>,187507,181,5990,applicant_race_token_set,other_nonamortizing_features,0.0965,3.1945,5
242,<NA>,187507,248,5990,applicant_race_token_set,negative_amortization,0.1323,3.1945,5
468,<NA>,187507,36050,5990,applicant_race_token_set,interest_only_payment,19.2259,3.1945,5
15,1,164764,174,4659,applicant_sex,other_nonamortizing_features,0.1056,2.8277,NaN
0,1,164764,192,4659,applicant_sex,negative_amortization,0.1165,2.8277,NaN
10,1,164764,1078,4659,applicant_sex,balloon_payment,0.6543,2.8277,NaN
5,1,164764,31973,4659,applicant_sex,interest_only_payment,19.4053,2.8277,NaN
6,2,102299,16763,2905,applicant_sex,interest_only_payment,16.3863,2.8397,NaN
1,2,102299,190,2905,applicant_sex,negative_amortization,0.1857,2.8397,NaN


,feature_or_block,timing_status,timing_evidence,quality_evidence,conditional_applicability,redundancy_evidence,organization_proxy_evidence,protected_group_sensitivity,screening_status,decision,decision_reason,gsv_player
0,negative_amortization,available_before_decision_by_definition,2024 HMDA FIG 4(a)(27): contractual terms incl...,100.00% valid; 97.05% informative; 0.14% code-...,All-cohort reporting field; code 1111 retained...,Maximum pairwise V: full=0.7072; informative-o...,LEI row-weighted dominance 99.95%; global 96.9...,not_applicable_excluded_before_modeling,exclude_from_further_review,exclude,Excluded for extreme rarity and concentration ...,not_applicable
1,interest_only_payment,available_before_decision_by_definition,2024 HMDA FIG 4(a)(27): contractual terms incl...,100.00% valid; 97.05% informative; 19.29% code...,All-cohort reporting field; code 1111 retained...,Maximum pairwise V: full=0.7173; informative-o...,LEI row-weighted dominance 89.37%; global 77.7...,Descriptive sex and canonical race-token-set s...,advance_to_grouped_validation_as_joint_block,<NA>,Final keep/exclude and player mapping are rese...,<NA>
2,balloon_payment,available_before_decision_by_definition,2024 HMDA FIG 4(a)(27): contractual terms incl...,100.00% valid; 97.05% informative; 2.37% code-...,All-cohort reporting field; code 1111 retained...,Maximum pairwise V: full=0.7173; informative-o...,LEI row-weighted dominance 98.34%; global 94.6...,Descriptive sex and canonical race-token-set s...,advance_to_grouped_validation_as_joint_block,<NA>,Final keep/exclude and player mapping are rese...,<NA>
3,other_nonamortizing_features,available_before_decision_by_definition,2024 HMDA FIG 4(a)(27): contractual terms incl...,100.00% valid; 97.05% informative; 0.14% code-...,All-cohort reporting field; code 1111 retained...,Maximum pairwise V: full=0.7081; informative-o...,LEI row-weighted dominance 99.83%; global 96.9...,not_applicable_excluded_before_modeling,exclude_from_further_review,exclude,Excluded for extreme rarity and concentration ...,not_applicable


prior_5_1_gate                True
schema_gate                   True
timing_gate                   True
informative_overlap_gate      True
lei_diagnostic_gate           True
protected_sensitivity_gate    True
screening_record_gate         True
ready_to_close_5_2            True
ready_to_start_5_3            True
dtype: bool

### 5.2.1. Kết luận screening

**Observed facts** — Bốn field có valid-code coverage 100%; code `1` xuất hiện ở 0,14% (`negative_amortization`), 19,29% (`interest_only_payment`), 2,37% (`balloon_payment`) và 0,14% (`other_nonamortizing_features`). Cả bốn cùng có 8.644 dòng `1111`, làm Cramér's V toàn cohort khoảng 0,71; trên informative-only population, association cao nhất là `interest_only_payment` × `balloon_payment` (V=0,1701).

**Interpretation** — `1111` là reporting exemption, không phải "No". Association được đọc trên full cohort và informative-only population để tách ảnh hưởng của exemption chung. LEI và protected metadata chỉ là diagnostic guardrails, không là predictors hoặc quy tắc tự động loại feature.

**Screening outcome** — `negative_amortization` (401 code-1 rows tại 4 LEIs; 392 tại một LEI) và `other_nonamortizing_features` (411 code-1 rows tại 18 LEIs) bị `exclude_from_further_review` vì rarity và concentration theo tổ chức. `interest_only_payment` + `balloon_payment` đi tiếp như joint candidate block; final keep/exclude và GSV player của hai field này vẫn để trống đến grouped train/validation sau khi 5.1–5.5 được screened. Không có kết luận về model performance, feature importance hoặc quan hệ nhân quả.


### 5.3. Manufactured home và multifamily — audited screening

**Phạm vi:** `manufactured_home_secured_property_type`, `manufactured_home_land_property_interest`, và `multifamily_affordable_units` trên cohort `dataset_b` (`action_taken ∈ {1, 2, 3}`).

**Ranh giới:** special states được bảo toàn; không impute `Not Applicable` thành `0`; `lei`, `applicant_sex` và năm race slots chỉ là diagnostic metadata, không vào `X`. FIG xác nhận domain báo cáo. Prediction-time lineage được tách riêng và không tự suy ra từ domain validation.

In [20]:
# 5.3.1 Schema and numeric audit — raw tokens remain distinct
from math import isfinite

GROUP_5_3_COLUMNS = [
    "manufactured_home_secured_property_type",
    "manufactured_home_land_property_interest",
    "multifamily_affordable_units",
]
RACE_SLOT_COLUMNS_5_3 = [f"applicant_race-{slot}" for slot in range(1, 6)]
SCHEMA_5_3 = {
    "manufactured_home_secured_property_type": {
        "informative": {"1", "2"}, "not_applicable": {"3"}, "exempt": {"1111", "EXEMPT"},
        "fig_reference": "2024 HMDA FIG field 89",
    },
    "manufactured_home_land_property_interest": {
        "informative": {"1", "2", "3", "4"}, "not_applicable": {"5"}, "exempt": {"1111", "EXEMPT"},
        "fig_reference": "2024 HMDA FIG field 90",
    },
    "multifamily_affordable_units": {
        "informative": "nonnegative_integer", "not_applicable": {"NA"}, "exempt": {"1111", "EXEMPT"},
        "fig_reference": "2024 HMDA FIG field 92",
    },
}

SPECIAL_BLANK_5_3 = {"", "NAN", "NONE", "<NA>"}
def normalized_token_5_3(value):
    return str(value).strip().upper()

def status_5_3(value, feature):
    token = normalized_token_5_3(value)
    if pd.isna(value) or token in SPECIAL_BLANK_5_3:
        return "Blank"
    rule = SCHEMA_5_3[feature]
    if token in rule["exempt"]:
        return "Exempt"
    if token in rule["not_applicable"]:
        return "Not Applicable"
    if feature != "multifamily_affordable_units":
        return "Informative" if token in rule["informative"] else "Invalid"
    try:
        numeric_value = float(token)
    except ValueError:
        return "Invalid"
    return "Informative" if isfinite(numeric_value) and numeric_value >= 0 and numeric_value.is_integer() else "Invalid"

analysis_5_3 = dataset_b.copy()
for feature in GROUP_5_3_COLUMNS:
    analysis_5_3[f"{feature}_status"] = analysis_5_3[feature].map(lambda value: status_5_3(value, feature))

quality_rows_5_3 = []
for feature in GROUP_5_3_COLUMNS:
    counts = analysis_5_3[f"{feature}_status"].value_counts()
    quality_rows_5_3.append({
        "feature": feature,
        "fig_reference": SCHEMA_5_3[feature]["fig_reference"],
        "row_count": len(analysis_5_3),
        "informative_n": int(counts.get("Informative", 0)),
        "not_applicable_n": int(counts.get("Not Applicable", 0)),
        "exempt_n": int(counts.get("Exempt", 0)),
        "blank_n": int(counts.get("Blank", 0)),
        "invalid_n": int(counts.get("Invalid", 0)),
    })
quality_5_3 = pd.DataFrame(quality_rows_5_3)
quality_5_3["reconciled_n"] = quality_5_3[["informative_n", "not_applicable_n", "exempt_n", "blank_n", "invalid_n"]].sum(axis=1)
quality_validation_5_3 = pd.Series({
    "cohort_is_dataset_b": len(analysis_5_3) == len(dataset_b),
    "all_counts_reconciled": quality_5_3["reconciled_n"].eq(len(dataset_b)).all(),
    "no_invalid_tokens": quality_5_3["invalid_n"].eq(0).all(),
    "no_blank_tokens": quality_5_3["blank_n"].eq(0).all(),
})
assert quality_validation_5_3[["cohort_is_dataset_b", "all_counts_reconciled"]].all()
display(quality_5_3)
display(quality_validation_5_3)

,feature,fig_reference,row_count,informative_n,not_applicable_n,exempt_n,blank_n,invalid_n,reconciled_n
0,manufactured_home_secured_property_type,2024 HMDA FIG field 89,293301,6262,278463,8576,0,0,293301
1,manufactured_home_land_property_interest,2024 HMDA FIG field 90,293301,6258,278424,8619,0,0,293301
2,multifamily_affordable_units,2024 HMDA FIG field 92,293301,2434,282219,8648,0,0,293301


cohort_is_dataset_b      True
all_counts_reconciled    True
no_invalid_tokens        True
no_blank_tokens          True
dtype: bool

In [21]:
# 5.3.2 Applicability audit — context first, then observed status
MULTIFAMILY_TOTAL_UNIT_TOKENS_5_3 = {"5-24", "25-49", "50-99", "100-149", ">149"}
analysis_5_3["is_manufactured_context"] = analysis_5_3["construction_method"].astype(str).str.strip().eq("2")
analysis_5_3["is_multifamily_context"] = analysis_5_3["total_units"].astype(str).str.strip().isin(MULTIFAMILY_TOTAL_UNIT_TOKENS_5_3)

applicability_specs_5_3 = [
    ("manufactured_home_secured_property_type", "is_manufactured_context"),
    ("manufactured_home_land_property_interest", "is_manufactured_context"),
    ("multifamily_affordable_units", "is_multifamily_context"),
]
applicability_rows_5_3 = []
for feature, context_column in applicability_specs_5_3:
    status_column = f"{feature}_status"
    in_context = analysis_5_3[context_column]
    statuses = analysis_5_3[status_column]
    applicability_rows_5_3.append({
        "feature": feature,
        "context_column": context_column,
        "applicable_context_n": int(in_context.sum()),
        "outside_context_n": int((~in_context).sum()),
        "outside_context_informative_n": int((~in_context & statuses.eq("Informative")).sum()),
        "in_context_not_applicable_n": int((in_context & statuses.eq("Not Applicable")).sum()),
        "in_context_exempt_n": int((in_context & statuses.eq("Exempt")).sum()),
        "in_context_blank_n": int((in_context & statuses.eq("Blank")).sum()),
    })
applicability_5_3 = pd.DataFrame(applicability_rows_5_3)

manufactured_cross_tabs_5_3 = {
    feature: pd.crosstab(
        analysis_5_3["construction_method"],
        analysis_5_3[f"{feature}_status"],
        margins=True,
    )
    for feature in ["manufactured_home_secured_property_type", "manufactured_home_land_property_interest"]
}
multifamily_cross_tab_5_3 = pd.crosstab(
    analysis_5_3["total_units"],
    analysis_5_3["multifamily_affordable_units_status"],
    margins=True,
)
applicability_validation_5_3 = pd.Series({
    "no_outside_context_informative_values": applicability_5_3["outside_context_informative_n"].eq(0).all(),
    "context_exceptions_recorded": applicability_5_3[["in_context_not_applicable_n", "in_context_exempt_n", "in_context_blank_n"]].notna().all().all(),
})
display(applicability_5_3)
display(applicability_validation_5_3)
for feature, table in manufactured_cross_tabs_5_3.items():
    print(feature)
    display(table)
display(multifamily_cross_tab_5_3)

,feature,context_column,applicable_context_n,outside_context_n,outside_context_informative_n,in_context_not_applicable_n,in_context_exempt_n,in_context_blank_n
0,manufactured_home_secured_property_type,is_manufactured_context,6815,286486,0,26,527,0
1,manufactured_home_land_property_interest,is_manufactured_context,6815,286486,0,28,529,0
2,multifamily_affordable_units,is_multifamily_context,3107,290194,0,163,510,0


no_outside_context_informative_values    True
context_exceptions_recorded              True
dtype: bool

manufactured_home_secured_property_type


manufactured_home_secured_property_type_status,Exempt,Informative,Not Applicable,All
construction_method,,,,
1,8049,0,278437,286486
2,527,6262,26,6815
All,8576,6262,278463,293301


manufactured_home_land_property_interest


manufactured_home_land_property_interest_status,Exempt,Informative,Not Applicable,All
construction_method,,,,
1,8090,0,278396,286486
2,529,6258,28,6815
All,8619,6258,278424,293301


multifamily_affordable_units_status,Exempt,Informative,Not Applicable,All
total_units,,,,
1,7152,0,252632,259784
100-149,17,88,3,108
2,723,0,23182,23905
25-49,57,342,6,405
3,148,0,4386,4534
4,115,0,1856,1971
5-24,373,1579,139,2091
50-99,50,278,5,333
>149,13,147,10,170


In [22]:
# 5.3.3 Targeted redundancy and numeric-distribution evidence
def cramers_v_5_3(table):
    table = table.loc[table.sum(axis=1).gt(0), table.sum(axis=0).gt(0)]
    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan
    observed = table.to_numpy(dtype=float)
    expected = np.outer(observed.sum(axis=1), observed.sum(axis=0)) / observed.sum()
    chi2 = ((observed - expected) ** 2 / expected).sum()
    return float(np.sqrt((chi2 / observed.sum()) / min(table.shape[0] - 1, table.shape[1] - 1)))

manufactured_data_5_3 = analysis_5_3.loc[analysis_5_3["is_manufactured_context"]].copy()
mfg_status_table_5_3 = pd.crosstab(
    manufactured_data_5_3["manufactured_home_secured_property_type_status"],
    manufactured_data_5_3["manufactured_home_land_property_interest_status"],
)
mfg_informative_data_5_3 = manufactured_data_5_3.loc[
    manufactured_data_5_3["manufactured_home_secured_property_type_status"].eq("Informative")
    & manufactured_data_5_3["manufactured_home_land_property_interest_status"].eq("Informative")
]
mfg_informative_table_5_3 = pd.crosstab(
    mfg_informative_data_5_3["manufactured_home_secured_property_type"],
    mfg_informative_data_5_3["manufactured_home_land_property_interest"],
)
redundancy_evidence_5_3 = pd.DataFrame([
    {
        "block": "manufactured_home_fields",
        "population": "manufactured_context_statuses",
        "n": len(manufactured_data_5_3),
        "cramers_v": cramers_v_5_3(mfg_status_table_5_3),
    },
    {
        "block": "manufactured_home_fields",
        "population": "manufactured_context_informative_only",
        "n": len(mfg_informative_data_5_3),
        "cramers_v": cramers_v_5_3(mfg_informative_table_5_3),
    },
])

multifamily_data_5_3 = analysis_5_3.loc[analysis_5_3["is_multifamily_context"]].copy()
informative_multifamily_5_3 = multifamily_data_5_3.loc[
    multifamily_data_5_3["multifamily_affordable_units_status"].eq("Informative")
].copy()
informative_multifamily_5_3["affordable_units"] = pd.to_numeric(
    informative_multifamily_5_3["multifamily_affordable_units"], errors="coerce"
)
multifamily_value_summary_5_3 = (
    informative_multifamily_5_3.groupby("total_units", dropna=False)["affordable_units"]
    .agg(n="size", zero_n=lambda values: int(values.eq(0).sum()), median="median", max="max")
    .reset_index()
)
multifamily_numeric_validation_5_3 = pd.Series({
    "all_informative_values_parsed": informative_multifamily_5_3["affordable_units"].notna().all(),
    "all_informative_values_nonnegative": informative_multifamily_5_3["affordable_units"].ge(0).all(),
    "all_informative_values_integer": informative_multifamily_5_3["affordable_units"].mod(1).eq(0).all(),
})
display(mfg_status_table_5_3)
display(mfg_informative_table_5_3)
display(redundancy_evidence_5_3)
display(multifamily_value_summary_5_3)
display(multifamily_numeric_validation_5_3)

manufactured_home_land_property_interest_status,Exempt,Informative,Not Applicable
manufactured_home_secured_property_type_status,,,
Exempt,527,0,0
Informative,2,6258,2
Not Applicable,0,0,26


manufactured_home_land_property_interest,1,2,3,4
manufactured_home_secured_property_type,,,,
1,2427,10,6,0
2,517,41,3117,140


,block,population,n,cramers_v
0,manufactured_home_fields,manufactured_context_statuses,6815,0.980861
1,manufactured_home_fields,manufactured_context_informative_only,6258,0.839286


,total_units,n,zero_n,median,max
0,100-149,88,73,0.0,100
1,25-49,342,276,0.0,100
2,5-24,1579,1413,0.0,100
3,50-99,278,221,0.0,100
4,>149,147,108,0.0,100


all_informative_values_parsed         True
all_informative_values_nonnegative    True
all_informative_values_integer        True
dtype: bool

In [23]:
# 5.3.4 LEI and protected-metadata guardrails — descriptive only
def lei_diagnostic_5_3(feature, scope):
    status_column = f"{feature}_status"
    valid_lei_scope = scope.loc[scope["lei"].notna() & scope["lei"].astype(str).str.strip().ne("")].copy()
    diagnostic_value = valid_lei_scope[feature].where(
        valid_lei_scope[status_column].eq("Informative"), valid_lei_scope[status_column]
    )
    global_dominance = diagnostic_value.value_counts(dropna=False, normalize=True).iloc[0] * 100
    per_lei = pd.DataFrame({"lei": valid_lei_scope["lei"], "value": diagnostic_value}).groupby("lei")["value"].agg(
        row_n="size", dominant_share=lambda values: values.value_counts(normalize=True, dropna=False).iloc[0] * 100
    )
    informative_by_lei = valid_lei_scope.loc[valid_lei_scope[status_column].eq("Informative")].groupby("lei").size()
    informative_n = int(informative_by_lei.sum())
    return {
        "feature": feature,
        "scope_n": len(scope),
        "lei_n": int(per_lei.shape[0]),
        "global_dominant_share_pct": round(global_dominance, 4),
        "row_weighted_dominant_share_pct": round(np.average(per_lei["dominant_share"], weights=per_lei["row_n"]), 4),
        "informative_lei_n": int(informative_by_lei.shape[0]),
        "informative_top_1_lei_share_pct": 0.0 if informative_n == 0 else round(informative_by_lei.nlargest(1).sum() / informative_n * 100, 4),
        "informative_top_10_lei_share_pct": 0.0 if informative_n == 0 else round(informative_by_lei.nlargest(10).sum() / informative_n * 100, 4),
    }

lei_proxy_5_3 = pd.DataFrame([
    lei_diagnostic_5_3("manufactured_home_secured_property_type", manufactured_data_5_3),
    lei_diagnostic_5_3("manufactured_home_land_property_interest", manufactured_data_5_3),
    lei_diagnostic_5_3("multifamily_affordable_units", multifamily_data_5_3),
])
lei_proxy_5_3["lei_uplift_vs_global_pp"] = (
    lei_proxy_5_3["row_weighted_dominant_share_pct"] - lei_proxy_5_3["global_dominant_share_pct"]
).round(4)

def canonical_race_set_5_3(row):
    tokens = sorted({
        normalized_token_5_3(row[column]) for column in RACE_SLOT_COLUMNS_5_3
        if normalized_token_5_3(row[column]) not in SPECIAL_BLANK_5_3
    })
    return "|".join(tokens) if tokens else "blank_or_missing"

analysis_5_3["applicant_race_token_set_5_3"] = analysis_5_3.apply(canonical_race_set_5_3, axis=1)
manufactured_data_5_3["applicant_race_token_set_5_3"] = manufactured_data_5_3.apply(canonical_race_set_5_3, axis=1)
multifamily_data_5_3["applicant_race_token_set_5_3"] = multifamily_data_5_3.apply(canonical_race_set_5_3, axis=1)
protected_rows_5_3 = []
for feature, scope in [
    ("manufactured_home_secured_property_type", manufactured_data_5_3),
    ("manufactured_home_land_property_interest", manufactured_data_5_3),
    ("multifamily_affordable_units", multifamily_data_5_3),
]:
    status_column = f"{feature}_status"
    for protected_group in ["applicant_sex", "applicant_race_token_set_5_3"]:
        grouped = scope.groupby(protected_group, dropna=False)[status_column]
        for group_value, statuses in grouped:
            protected_rows_5_3.append({
                "feature": feature,
                "protected_group": protected_group,
                "group_value": group_value,
                "n": int(statuses.size),
                "informative_pct": round(statuses.eq("Informative").mean() * 100, 4),
                "special_or_not_applicable_pct": round(statuses.isin(["Exempt", "Not Applicable", "Blank"]).mean() * 100, 4),
            })
protected_sensitivity_5_3 = pd.DataFrame(protected_rows_5_3)
assert lei_proxy_5_3["lei_uplift_vs_global_pp"].notna().all()
assert not protected_sensitivity_5_3.empty
display(lei_proxy_5_3)
display(protected_sensitivity_5_3.sort_values(["feature", "protected_group", "n"], ascending=[True, True, False]))

,feature,scope_n,lei_n,global_dominant_share_pct,row_weighted_dominant_share_pct,informative_lei_n,informative_top_1_lei_share_pct,informative_top_10_lei_share_pct,lei_uplift_vs_global_pp
0,manufactured_home_secured_property_type,6815,186,55.9941,93.3382,149,21.6385,74.9760,37.3441
1,manufactured_home_land_property_interest,6815,186,45.8254,86.9699,148,21.6523,75.0240,41.1445
2,multifamily_affordable_units,3107,215,67.2996,88.9282,141,14.3796,48.8085,21.6286


,feature,protected_group,group_value,n,informative_pct,special_or_not_applicable_pct
85,manufactured_home_land_property_interest,applicant_race_token_set_5_3,5,5532,91.1605,8.8395
86,manufactured_home_land_property_interest,applicant_race_token_set_5_3,6,846,96.9267,3.0733
76,manufactured_home_land_property_interest,applicant_race_token_set_5_3,3,179,96.6480,3.3520
87,manufactured_home_land_property_interest,applicant_race_token_set_5_3,7,79,62.0253,37.9747
49,manufactured_home_land_property_interest,applicant_race_token_set_5_3,1,67,94.0299,5.9701
...,...,...,...,...,...,...
91,multifamily_affordable_units,applicant_sex,4,2559,77.6866,22.3134
88,multifamily_affordable_units,applicant_sex,1,313,76.3578,23.6422
90,multifamily_affordable_units,applicant_sex,3,184,92.9348,7.0652
89,multifamily_affordable_units,applicant_sex,2,50,70.0000,30.0000


In [24]:
# 5.3.5 Evidence-based screening record and closure gate
# FIG fields 89, 90, and 92 validate the reporting domain. The local project has no
# source proving each value is available at the approval-model prediction point.
timing_evidence_5_3 = pd.DataFrame([
    {
        "feature_or_block": feature,
        "timing_status": "requires_lineage_confirmation",
        "timing_evidence": f"{SCHEMA_5_3[feature]['fig_reference']} validates report-domain values; it does not establish local prediction-time lineage.",
    }
    for feature in GROUP_5_3_COLUMNS
])

section_5_3_gate = pd.Series({
    "prior_5_2_gate": bool(section_5_2_gate["ready_to_start_5_3"]),
    "schema_gate": bool(quality_validation_5_3[["all_counts_reconciled", "no_invalid_tokens", "no_blank_tokens"]].all()),
    "numeric_gate": bool(multifamily_numeric_validation_5_3.all()),
    "applicability_gate": bool(applicability_validation_5_3["no_outside_context_informative_values"]),
    "applicability_exceptions_recorded": bool(applicability_validation_5_3["context_exceptions_recorded"]),
    "redundancy_evidence_gate": redundancy_evidence_5_3["cramers_v"].notna().all(),
    "lei_diagnostic_gate": lei_proxy_5_3["lei_uplift_vs_global_pp"].notna().all(),
    "protected_sensitivity_gate": not protected_sensitivity_5_3.empty,
    "timing_gate": timing_evidence_5_3["timing_status"].eq("available_before_decision_by_definition").all(),
})
section_5_3_gate["ready_to_close_5_3"] = bool(section_5_3_gate.all())
section_5_3_gate["ready_to_start_5_4"] = bool(section_5_3_gate["ready_to_close_5_3"])

screening_status_5_3 = (
    "advance_to_grouped_validation" if section_5_3_gate["ready_to_close_5_3"]
    else "hold_for_timing_lineage_confirmation"
)
screening_decision_5_3 = pd.DataFrame({"feature_or_block": GROUP_5_3_COLUMNS}).merge(
    timing_evidence_5_3, on="feature_or_block", validate="one_to_one"
)
screening_decision_5_3["business_reason"] = screening_decision_5_3["feature_or_block"].map({
    "manufactured_home_secured_property_type": "Property collateral context for manufactured-home applications.",
    "manufactured_home_land_property_interest": "Land-interest context for manufactured-home applications.",
    "multifamily_affordable_units": "Affordable-unit count for applicable multifamily properties.",
})
screening_decision_5_3["quality_evidence"] = screening_decision_5_3["feature_or_block"].map(
    quality_5_3.set_index("feature")["informative_n"].map(lambda n: f"{int(n)} informative rows; schema audit reconciled.")
)
screening_decision_5_3["redundancy_evidence"] = screening_decision_5_3["feature_or_block"].map({
    "manufactured_home_secured_property_type": "Reviewed jointly with manufactured_home_land_property_interest in manufactured context.",
    "manufactured_home_land_property_interest": "Reviewed jointly with manufactured_home_secured_property_type in manufactured context.",
    "multifamily_affordable_units": "Distribution reviewed only within multifamily context; raw-vs-derived dwelling comparison remains a separate representation decision.",
})
screening_decision_5_3["protected_group_sensitivity"] = "Coverage/status diagnostics recorded by applicant_sex and canonical five-slot race token set."
screening_decision_5_3["screening_status"] = screening_status_5_3
screening_decision_5_3["decision"] = pd.NA
screening_decision_5_3["decision_reason"] = "Final keep/exclude and GSV player are reserved until timing lineage is confirmed and grouped validation is authorized."
screening_decision_5_3["gsv_player"] = pd.NA

assert section_5_3_gate[["schema_gate", "numeric_gate", "applicability_gate", "lei_diagnostic_gate", "protected_sensitivity_gate"]].all()
assert screening_decision_5_3["screening_status"].notna().all()
assert screening_decision_5_3["decision"].isna().all()
assert screening_decision_5_3["gsv_player"].isna().all()
display(section_5_3_gate)
display(screening_decision_5_3)

prior_5_2_gate                        True
schema_gate                           True
numeric_gate                          True
applicability_gate                    True
applicability_exceptions_recorded     True
redundancy_evidence_gate              True
lei_diagnostic_gate                   True
protected_sensitivity_gate            True
timing_gate                          False
ready_to_close_5_3                   False
ready_to_start_5_4                   False
dtype: bool

,feature_or_block,timing_status,timing_evidence,business_reason,quality_evidence,redundancy_evidence,protected_group_sensitivity,screening_status,decision,decision_reason,gsv_player
0,manufactured_home_secured_property_type,requires_lineage_confirmation,2024 HMDA FIG field 89 validates report-domain...,Property collateral context for manufactured-h...,6262 informative rows; schema audit reconciled.,Reviewed jointly with manufactured_home_land_p...,Coverage/status diagnostics recorded by applic...,hold_for_timing_lineage_confirmation,<NA>,Final keep/exclude and GSV player are reserved...,<NA>
1,manufactured_home_land_property_interest,requires_lineage_confirmation,2024 HMDA FIG field 90 validates report-domain...,Land-interest context for manufactured-home ap...,6258 informative rows; schema audit reconciled.,Reviewed jointly with manufactured_home_secure...,Coverage/status diagnostics recorded by applic...,hold_for_timing_lineage_confirmation,<NA>,Final keep/exclude and GSV player are reserved...,<NA>
2,multifamily_affordable_units,requires_lineage_confirmation,2024 HMDA FIG field 92 validates report-domain...,Affordable-unit count for applicable multifami...,2434 informative rows; schema audit reconciled.,Distribution reviewed only within multifamily ...,Coverage/status diagnostics recorded by applic...,hold_for_timing_lineage_confirmation,<NA>,Final keep/exclude and GSV player are reserved...,<NA>


### 5.3.1 Kết luận screening

**Observed facts** — Cả ba fields reconcile đủ 293.301 dòng, không có blank hoặc invalid token. Informative counts lần lượt là 6.262 (`manufactured_home_secured_property_type`), 6.258 (`manufactured_home_land_property_interest`) và 2.434 (`multifamily_affordable_units`). Không có informative value ngoài applicable context. Các special/context exceptions được giữ riêng: 26, 28 và 163 dòng Not Applicable trong context tương ứng; 527, 529 và 510 dòng Exempt trong context tương ứng.

**Redundancy and guardrails** — Hai manufactured-home fields có Cramér's V = 0,9809 khi tính status trong manufactured context và 0,8393 khi chỉ tính informative values; vì vậy chúng chỉ được xem xét như một joint business block. Với 2.434 multifamily informative rows, mọi giá trị đều parse được, không âm và là integer; 2.091 dòng bằng 0. LEI row-weighted dominance vượt global dominance lần lượt 37,3441; 41,1445; và 21,6286 điểm phần trăm. Đây là diagnostic proxy-risk, không phải quy tắc tự động loại feature. Protected diagnostics dùng `applicant_sex` và canonical set của cả năm race slots, chỉ mô tả coverage/status.

**Screening outcome** — FIG fields 89, 90 và 92 xác minh reporting domain nhưng chưa chứng minh prediction-time lineage tại điểm dự báo của mô hình trong dự án. Vì `timing_gate=False`, cả ba fields có trạng thái `hold_for_timing_lineage_confirmation`; `decision` và `gsv_player` vẫn để trống. 5.3 chưa được đóng và không khởi động 5.4. Không có kết luận về model performance, feature importance, fairness outcome hoặc quan hệ nhân quả.